In [1]:
import os
os.chdir('/workspace/97f64178-201a-43bc-9926-395b3117c761')
print(os.listdir('.'))


['.config', 'zeta_zeros_N5000_dps50.npy', 'ldh_zeros_N5000_dps50.npy', '-PROMPT-v6-DATASET.md', '.kernel_llm_logs_1.txt', '.prompts', 'lchi5_zeros_N5000_dps80.npy', 'memory', 'ldh_off_line_zeros.csv']


In [2]:
import numpy as np, pandas as pd, mpmath as mp
from mpmath import mpf, mpc, mp as mpctx

zeta = np.load('zeta_zeros_N5000_dps50.npy')
ldh = np.load('ldh_zeros_N5000_dps50.npy')
lchi = np.load('lchi5_zeros_N5000_dps80.npy')
offline = pd.read_csv('ldh_off_line_zeros.csv')
print(zeta.shape, zeta.dtype, zeta[0])
print(ldh.shape, ldh.dtype, ldh[0])
print(lchi.shape, lchi.dtype, lchi[0])
print(offline.shape, offline.columns.tolist())
print(offline.head(3))


(5000,) <U51 14.134725141734693790457251983562470270784257115699
(5000,) <U51 5.0941598445710949256987955170797974750670744531091
(5000,) <U81 6.6484533447277147161232784599793178472985854232444983723747129467
(110, 5) ['sigma', 't', 'sigma_str', 't_str', 'absL']
 sigma t sigma_str t_str absL
0 0.650830 114.163343 0.650830 114.163343 8.093347e-51
1 0.574356 166.479306 0.574356 166.479306 8.001542e-50
2 0.869531 240.404672 0.869531 240.404672 1.130345e-49


In [3]:
# Plan:
# We'll implement Weil's explicit formula as a quadratic form on test functions.
# 
# Standard Weil explicit formula for an L-function L(s) with completed function Λ(s)=γ(s) L(s) (functional equation Λ(s)=ε Λ(1-s)):
# For a suitable even Schwartz test function F on R with Fourier transform Φ(u)= ∫ F(t) e^{-i t u} dt (or pair Φ↔F appropriately),
# 
# Σ_ρ F(γ_ρ) = (archimedean term involving γ-factors) + (prime sum involving Λ_L(n) Φ(log n)) + pole terms.
# 
# For our PURPOSES (positivity tests), we use the symmetric quadratic-form version:
# W(F) = Σ_ρ F(γ_ρ) - A_∞(F) - P(F)
# 
# If RH holds, all γ_ρ are real, and Weil positivity states that for F = |Φ|^2 with Φ even Schwartz,
# W(F) ≥ 0 iff GRH holds (in fact Weil's criterion).
# 
# We choose F_j(t) = sinc-type even functions whose Fourier transforms have compact support in [-X,X].
# A convenient choice: Let h(u)= 1_{|u|<=X}, then F = (h*h)(t)/(2π) ... easier to use:
# 
# Standard convention (Iwaniec-Kowalski): Let φ be even Schwartz, Φ(s)=∫ φ(x) e^{(s-1/2)x} dx.
# Then the explicit formula reads:
# Σ_ρ Φ(ρ) = Φ(0)+Φ(1) - Σ_{n} Λ(n) (a_n n^{-1/2}) (φ(log n) + φ̄(log n)?) etc.
# 
# This is complex. We'll use a cleaner setup using even functions of t on critical line:
#
# Let f: R -> R be even Schwartz, and let g(u)=∫ f(t) cos(tu) dt (Fourier cosine transform / 2).
# Weil's explicit formula (Bombieri/Mestre form) for an L-function with completed Λ(s)=Q^s · ∏Γ((s+μ_j)/2) · L(s):
#
# Σ_ρ f(γ_ρ) = (1/π) ∫_{-∞}^∞ f(t) [log Q + Σ_j Re ψ((1/2 + iμ_j' + it)/?... ] dt [archimedean]
# - 2 Σ_n (Λ_L(n)/√n) g(log n) [prime sum]
# + boundary (poles)
# 
# where ρ = 1/2 + i γ_ρ (real γ_ρ under RH).
# 
# This is the explicit formula in the symmetric form due to Weil (cf. https://en.wikipedia.org/wiki/Weil%27s_explicit_formula).
# 
# Concretely, taking the symmetric form for the Riemann zeta function:
#
# Σ_ρ f(γ_ρ) = 2 ∫_0^∞ f(t) [Re ψ(1/4 + it/2) - log π] dt /(2π) -- need to verify
# - 2 Σ_n Λ(n)/√n · g(log n)
# + f(i/2) + f(-i/2)
# 
# To avoid pitfalls in coefficient conventions, we use a slightly different formulation:
# We test the FORMULA itself (not positivity) by checking it numerically holds for ζ on a known test function — actually for positivity test we just want sign of λ_min.
# 
# Let me adopt the formulation from Bombieri's "Remarks on Weil's quadratic functional in the theory of prime numbers I":
# 
# For F continuous on R, even, compactly supported, define
# f(s) = ∫_{-∞}^∞ F(x) e^{(s-1/2)x} dx
# Then explicit formula for ζ (Riemann–von Mangoldt explicit formula in symmetric form):
# Σ_ρ f(ρ) = f(0)+f(1) - 2 Σ_{n≥1} Λ(n) F(log n) n^{-1/2}? ...
# 
# Hmm. The cleanest is to use the "FORM" already coded in numbers theory literature for positivity tests. 
# I'll implement the version from Bombieri (1997) precisely as used in the Bombieri–Lagarias positivity tests.
print("ok")


ok


In [4]:
# I'll use the Guinand-Weil explicit formula in the convenient form for an L-function L(s) with
# functional equation Λ(s) = ε · q^{s/2} ·∏_j Γ((s+κ_j)/2) · L(s) = ε̄ ·Λ(1-s).
# Here q is the conductor, κ_j the gamma factor parameters.
#
# Let h: R -> R be EVEN Schwartz, with Fourier transform ĥ(u) = ∫ h(t) e^{-itu} dt.
# Define the Weil functional:
# W_L(h) := Σ_ρ h(γ_ρ) - A_∞(h) - P(h),
# 
# where (writing ρ = β + iγ; under GRH β=1/2):
# A_∞(h) = (1/(2π)) ∫_{-∞}^{∞} h(t) · [ log q + Σ_j Re ψ(1/4 + κ_j/2 + i t/2 ) ] dt
# P(h) = (1/(2π)) · ( 2 Σ_n (Λ_L(n)/√n) ĥ(log n) )
# 
# Reference: Iwaniec-Kowalski Theorem 5.12.
# Actually: Σ_ρ h(γ_ρ) = (1/(2π)) ∫ h(t)·[2 log q^{1/2} + Σ_j (Γ'/Γ)(1/4+κ_j/2 + it/2)] dt 
# -(1/π) Σ_n Λ_L(n) ĥ(log n)/√n (Plus pole terms if L has pole)
# 
# Where ĥ(ξ) = ∫ h(t) e^{-itξ} dt = ∫ h(t) cos(tξ) dt (since h even).
# 
# For POSITIVITY: choose h(t) = |g(t)|² where g is real-valued even. Equivalently h is the Fourier convolution g*g̃.
# Then Weil positivity: GRH ⇔ W_L(h) ≥ 0 for all such h.
# 
# **Bilinear form**: Define bilinear pairing ⟨g, g'⟩_L := W_L(... ) obtained by polarization. For real-valued g, g':
# B_L(g, g') = Σ_ρ ĝ(γ_ρ) ĝ'(γ_ρ) - Archimedean(g,g') - Prime(g,g')
# 
# (since the Fourier transform of g*g̃ is |ĝ|^2; and bilinear extension is ĝ · ĝ').
# This is the cleanest version for matrix construction.
#
# So the matrix entries are:
# M_jk = Σ_ρ ĝ_j(γ_ρ) ĝ_k(γ_ρ) - Arch(g_j, g_k) - Prime(g_j, g_k)
# 
# Specifically picking g_j with compactly supported Fourier transforms ĝ_j, the prime sum is finite.
# So choose ĝ_j(u) = even bump in Fourier space supported in [-X, X], which means g_j(t) is the inverse transform.
# 
# Actually for the QUADRATIC FORM and explicit formula, we want g_j to be even Schwartz with COMPACTLY SUPPORTED Fourier transform ĝ_j.
# Then:
# - the zero sum Σ_ρ ĝ_j(γ_ρ) ĝ_k(γ_ρ) is a finite or rapidly-convergent sum,
# - the prime sum Σ_n Λ(n)/√n · (g_j*g_k... ) involves only primes with log p ≤ supp,
# - the archimedean term is a one-dim integral over R of h_jk(t) times log|conductor| + digamma terms.
# 
# Wait — I conflated h and g. Let me redo:
# 
# Take g_j: R -> R even Schwartz. Let h_{jk}(t) = g_j(t) g_k(t) (pointwise product). This is also even Schwartz.
# Its Fourier transform is ĥ_{jk}(u) = (ĝ_j * ĝ_k)(u)/(2π) [convolution].
# 
# Weil formula applied to h_{jk}:
# Σ_ρ h_{jk}(γ_ρ) = Arch(h_{jk}) + Prime(h_{jk})
# 
# So M_jk = Σ_ρ g_j(γ_ρ) g_k(γ_ρ) − Arch(g_j g_k) − Prime(g_j g_k).
# 
# Quadratic form Q(c) = Σ_jk c_j c_k M_jk = W_L( (Σ c_j g_j)^2 ) ≥ 0 under GRH.
# 
# Excellent. We need ĝ_j to have compact support so the prime sum is finite, but g_j (and its product) need not.
# 
# CHOICE: Let g_j(t) = sin(X t / 2 + j π/2)/(t) variants? Better: pick ĝ_j(u) = piecewise-linear bumps.
# Simplest: use Fourier characters of a Paley-Wiener basis. Take ĝ_j(u) = 1_{[-X,X]}(u) · cos(j π u/X) (chunks).
# Then g_j(t) = (X/π) sinc(...) — bandlimited functions.
#
# To keep this MANAGEABLE: I'll use a simple basis of bandlimited test functions
# g_j(t) := (1/(2π)) ∫_{-X}^{X} ĝ_j(u) e^{itu} du = (1/(2π))∫_{-X}^X ĝ_j(u) cos(tu) du [since ĝ_j even].
# 
# Choose ĝ_j(u) = T_j(u/X) · 1_{|u|≤X} where T_j is even Chebyshev, j=0,2,4,...
# Or simpler: use Hermite-like or just take indicator/triangle.
# 
# Plan: use BASIS B_j(u) = (1 - |u|/X) · cos(2π j u / (2X))·1_{|u|≤X} (modulated triangle). These give bandlimited even g_j.
#
# Let me FIX X=2 (so log p ≤ 2 → primes 2,3,5,7), and use a small basis (e.g., 4-6 functions). Then compute the matrix.
print("design ok")


design ok


In [5]:
# Let me carefully nail down the Weil explicit formula. I'll follow Iwaniec-Kowalski Theorem 5.12.
#
# For an L-function in the Selberg class with completed L-function
# Λ(s) = q^{s/2} γ(s) L(s), γ(s) = ∏_{j=1}^d Γ((s+κ_j)/2),
# functional equation Λ(s) = ε · \overline{Λ(1-\bar s)}.
#
# Let φ be even, holomorphic in a horizontal strip |Im s| < 1/2 + δ, and decay like (1+|t|)^{-2-δ}.
# Let φ̂(u) = (1/(2π)) ∫ φ(t) e^{-iut} dt (so φ(t) = ∫ φ̂(u) e^{iut} du).
# 
# Then Iwaniec-Kowalski (Theorem 5.12):
# Σ_ρ φ(γ_ρ) = φ(i/2) + φ(-i/2) [pole terms; only for ζ which has pole at s=1]
# + (log q)/(2π) ∫_R φ(t) dt
# + (1/(2π)) ∫_R φ(t) · Σ_j Re[ψ(1/4 + κ_j/2 + it/2)] dt ... HMMMM no.
# - Σ_n (Λ_L(n)/√n) [φ̂(log n) + φ̂(-log n)]·??
# 
# Honestly, I want to be careful. Let me use a reference-quality formulation.
#
# I'll use the form in Mestre (1986) / Bombieri-Lagarias (1999) "Complements to Li's Criterion for the Riemann Hypothesis":
# 
# Define for even real F ∈ S(R), with Fourier pair F̂(ξ) = ∫ F(x) e^{-2π i ξ x} dx (standard convention),
# 
# W_L(F) := Σ_γ F̂(γ/(2π)) [zero sum over γ where ρ=1/2+iγ]
# 
# equals
# F̂(i/2/(2π)) + F̂(-i/2/(2π)) [poles of L: present for ζ at s=1]
# + 2 ∫_0^∞ F̂(ξ) · K_∞(ξ) dξ [archimedean]
# - 2 Σ_{n} (Λ_L(n)/√n) F(log n / (2π)) [prime sum]
# 
# ... I keep tangling conventions. Let me **pick one convention and stick rigorously**.
#
# CONVENTION (mine, clear):
# Fourier pair: ĥ(u) = ∫_{-∞}^∞ h(t) e^{-i t u} dt, h(t) = (1/(2π)) ∫ ĥ(u) e^{i t u} du.
#
# Weil explicit formula (e.g., Rudnick-Sarnak 1996 "Zeros of Principal L-functions and Random Matrix Theory" eq (2.1) or thereabouts):
#
# For L(s) Selberg-class with Λ(s)=q^{s/2}∏_j Γ((s+κ_j)/2) L(s), functional eq Λ(s)=ε Λ̄(1-s):
# For even Schwartz φ:
# Σ_γ φ(γ) = (1/(2π)) ∫ φ(r) [2 log q^{1/2} + Σ_j (Γ'/Γ)(1/4 + κ_j/2 + ir/2)·(1) ... ] dr
# - (1/π) Σ_{n≥1} (Λ_L(n)/√n) φ̂(log n)
# + (poles: φ(-i/2)+φ(i/2) for L=ζ)
# 
# where the archimedean term uses Re of the digamma (since the formula is real for real even φ).
# 
# Specifically (Rudnick-Sarnak eq 2.5):
# Σ_γ φ(γ) = (1/(2π)) ∫_{-∞}^{∞} φ(r) Ω_L(r) dr - (1/π) Σ_n (Λ_L(n)/√n) φ̂(log n) + poles
# 
# where
# Ω_L(r) = log q + Σ_j [Γ'/Γ (1/4+κ_j/2+ir/2) + Γ'/Γ (1/4+κ̄_j/2-ir/2)] / 2 ... aka Re of digamma sum.
#
# I'll trust this form. Numerical verification will be done on ζ alone with a specific φ to confirm signs/constants.
#
# **Verification strategy**: Pick φ(t)=exp(-t^2/2). Then φ̂(u)=√(2π) exp(-u²/2).
# We'll compute LHS = Σ_γ φ(γ_n) for ζ (over the 5000 known zeros) and RHS, and check they match.
print("plan v2")


plan v2


In [6]:
# Numerical sanity check first for zeta.
# I'll convert all zero data to floats (sufficient precision for these matrix tests; the 14-decimal precision in zeros is plenty).
import numpy as np
import mpmath as mp
from scipy.special import digamma
import math

zeta_g = np.array([float(s) for s in zeta])
ldh_g = np.array([float(s) for s in ldh])
lchi_g = np.array([float(s) for s in lchi])

# For L_DH off-line zeros, we have (sigma, t) pairs with sigma ≠ 1/2.
offline_arr = offline[['sigma','t']].to_numpy(dtype=float)
print("counts:", len(zeta_g), len(ldh_g), len(lchi_g), len(offline_arr))
print("zeta first/last:", zeta_g[0], zeta_g[-1])
print("ldh first/last:", ldh_g[0], ldh_g[-1])
print("lchi first/last:", lchi_g[0], lchi_g[-1])
print("offline sigma range:", offline_arr[:,0].min(), offline_arr[:,0].max())
print("offline t range:", offline_arr[:,1].min(), offline_arr[:,1].max())


counts: 5000 5000 5000 110
zeta first/last: 14.134725141734695 5447.8619983012995
ldh first/last: 5.094159844571095 4981.167489862579
lchi first/last: 6.648453344727715 4388.20901259418
offline sigma range: 0.5159183143809815 0.8977501549770109
offline t range: 114.16334273075698 4999.5661784847725


In [7]:
# Numerical sanity check of Weil explicit formula for zeta with Gaussian test function.
# 
# Form: Σ_γ φ(γ) = (1/(2π)) ∫ φ(r) Ω(r) dr - (1/π) Σ_n Λ(n)/√n · φ̂(log n) + φ(i/2)+φ(-i/2)
# 
# For ζ: q=1, single gamma factor Γ(s/2), so κ_1=0. Thus log q = 0. 
# Ω_ζ(r) = -log π + Re[ψ(1/4 + ir/2)] (from log of π^{-s/2} Γ(s/2) → derivative).
# Wait. The completed zeta is ξ(s) = π^{-s/2} Γ(s/2) ζ(s). So q^{s/2} where q=1, and we have a factor π^{-s/2}.
# Standard explicit formula uses (Γ'/Γ)(s/2) etc, but the π^{-s/2} contributes -log π.
# 
# Actually the conductor for ζ is 1 but you can think of it as π for archimedean reasons. Let me just write the precise formula for ζ:
# 
# Σ_γ φ(γ) = (1/(2π)) ∫ φ(r) · [-log π + Re ψ(1/4 + ir/2)] dr · 2 ?? hmm.
# 
# Let me just look up directly. From standard textbook (Titchmarsh, Iwaniec-Kowalski), the explicit formula for ζ in this form:
# 
# Σ_γ h(γ) = 2 h(i/2) + (1/(2π)) ∫_{-∞}^{∞} h(r) Ψ(r) dr - 2 Σ_n (Λ(n)/√n) g(log n)
# 
# where h is even, g(x) = (1/(2π)) ∫ h(r) e^{-irx} dr (so h = ĝ in the standard sense, g(x)=ĥ(x)/(2π)? careful).
# And Ψ(r) = Re [Γ'/Γ(1/4 + ir/2)] - log π.
# 
# Here "2 h(i/2)" comes from pole at s=1 (and trivially also from s=0 by FE, but ξ has trivial zeros that net to "0" — actually ζ explicit gives pole contribution at s=1 and s=0).
# 
# I'll **verify numerically** for h(r) = e^{-r²/2}, g(x)= (1/(2π))∫ e^{-r²/2} e^{-irx} dr = (1/√(2π)) e^{-x²/2}.
# (Verify: ∫ e^{-r²/2} cos(rx) dr = √(2π) e^{-x²/2}, so g(x) = √(2π)/(2π) e^{-x²/2} = e^{-x²/2}/√(2π) ✓)

import sympy as sp
from sympy import primerange
from mpmath import digamma as mdigamma

# LHS: zero sum for ζ
zeta_g_mp = [mp.mpf(s) for s in zeta]
mp.mp.dps = 30

def lhs_zeta(h):
 # h: function of mp.mpf
 s = mp.mpf(0)
 for g in zeta_g_mp:
 s += h(g)
 return 2*s # zeros come in ±γ pairs, sum extends symmetrically

# Gaussian h
def h_gauss(r):
 return mp.exp(-r**2 / 2)

# Compute LHS with N=5000 (well, 2*5000)
lhs = lhs_zeta(h_gauss)
print("LHS (Σ over ±γ for 10000 zeros):", lhs)

# Archimedean integral: (1/(2π)) ∫ h(r) [-log π + Re ψ(1/4 + ir/2)] dr
def arch_integrand(r):
 return h_gauss(r) * (mp.re(mdigamma(mp.mpf('0.25') + 1j*r/2)) - mp.log(mp.pi))
arch = mp.quad(arch_integrand, [-50, 50]) / (2*mp.pi)
print("Archimedean term:", arch)

# Prime sum: 2 Σ_n Λ(n)/√n · g(log n), with g(x) = e^{-x²/2}/√(2π)
def g_gauss(x):
 return mp.exp(-x**2/2) / mp.sqrt(2*mp.pi)

# Λ(n)=log p if n=p^k, 0 otherwise. Need this for n s.t. g(log n) is significant.
# g(log n) ~ exp(-(log n)^2/2) — significant for log n ≤ ~10, i.e., n ≤ e^10 ≈ 22000.
from sympy import factorint, isprime
prime_sum = mp.mpf(0)
for n in range(2, 30000):
 fac = factorint(n)
 if len(fac)==1: # prime power
 p, k = next(iter(fac.items()))
 Lambda = mp.log(p)
 prime_sum += Lambda / mp.sqrt(n) * g_gauss(mp.log(n))
prime_sum_term = 2 * prime_sum
print("Prime sum (term to subtract):", prime_sum_term)

# Pole contributions: 2 h(i/2) = 2 e^{(i/2)^2 /(-2)}? h(r)=e^{-r²/2}. h(i/2)=e^{-(i/2)²/2}= e^{-(-1/4)/2}=e^{1/8}.
pole = 2 * mp.exp(mp.mpf(1)/8)
print("Pole term 2 h(i/2):", pole)

# RHS = pole + arch - prime
rhs = pole + arch - prime_sum_term
print("RHS:", rhs)
print("LHS - RHS:", lhs - rhs)


LHS (Σ over ±γ for 10000 zeros): 8.26197659486072438968955962622e-44


Archimedean term: -1.20997344840977093249162415954


Prime sum (term to subtract): 1.06995018805559480827095954664
Pole term 2 h(i/2): 2.26629690613365263365801445562
RHS: -0.0136267303317131071045692505511
LHS - RHS: 0.0136267303317131071045692505511


In [8]:
# LHS is essentially zero! That means my h(r)=e^{-r²/2} decays so fast that Σ over γ for |γ|>~30 is negligible — and h(γ_1=14.13)≈e^{-100}≈0.
# Wait, h(14.13)=e^{-14.13²/2}=e^{-99.8}. Yes, machine epsilon-level. So LHS≈0.
# 
# That means my Gaussian is way too narrow. Need wider one.
def h_gauss(r, sigma=20.0):
 return mp.exp(-r**2/(2*sigma**2))
sigma = 20.0
# LHS
s = mp.mpf(0)
for g in zeta_g_mp[:5000]:
 s += h_gauss(g, sigma)
lhs = 2*s
print("LHS:", lhs)

# Note: g(x) here = (1/(2π)) ∫ h(r) e^{-irx} dr = (σ/√(2π)) e^{-σ²x²/2}
def g_gauss(x, sigma=20.0):
 return (sigma/mp.sqrt(2*mp.pi)) * mp.exp(-(sigma*x)**2/2)

# Arch
def arch_integrand(r):
 return h_gauss(r, sigma) * (mp.re(mdigamma(mp.mpf('0.25') + 1j*r/2)) - mp.log(mp.pi))
arch = mp.quad(arch_integrand, [-100, 100]) / (2*mp.pi)
print("Arch:", arch)

# Prime sum: σ=20 means g(x) decays as e^{-200 x²}, so log n must be small (<~ 0.3). Only n=2 contributes
# Actually with σ=20, g(log 2)= (20/√(2π)) exp(-(20·log2)²/2)= 20/√(2π) · exp(-(13.86)²/2)=20/√(2π)·exp(-96)
# This is tiny. So prime sum is tiny.
prime_sum = mp.mpf(0)
for n in range(2, 100):
 fac = factorint(n)
 if len(fac)==1:
 p, k = next(iter(fac.items()))
 Lambda = mp.log(p)
 prime_sum += Lambda / mp.sqrt(n) * g_gauss(mp.log(n), sigma)
print("Prime sum:", 2*prime_sum)

pole = 2 * h_gauss(mp.mpc(0, mp.mpf('0.5')), sigma) # h(i/2) = exp(-(i/2)²/(2σ²)) = exp(1/(8σ²))
print("Pole:", pole)

rhs = pole + arch - 2*prime_sum
print("RHS:", rhs)
print("LHS-RHS:", lhs - rhs)


LHS: 5.92186759948766021405387441401


Arch: 3.75360575370256606853827526873
Prime sum: 1.45098181220793866323122426626e-41
Pole: (2.00062509766642332081993679631 + 0.0j)
RHS: (5.75423085136898938935821206504 + 0.0j)
LHS-RHS: (0.16763674811867082469566234897 + 0.0j)


In [9]:
# Off by ~0.17. Hmm. Either truncation of zero sum (need more zeros — at γ_5000 ~ 5447, but with σ=20, h(5447)=e^{-37000}~0, irrelevant), 
# OR archimedean term truncation (integrate to ±100 — for σ=20, h(100)=e^{-12.5}≈3.7e-6, may matter due to digamma growth)
# OR formula constants are off.
# Let me extend the integration range and check.
arch2 = mp.quad(arch_integrand, [-500, 500]) / (2*mp.pi)
print("Arch wider:", arch2)


Arch wider: 0.736413822026978247080698052905


In [10]:
# Huge change — the integrand decays slowly because digamma grows like log(r) so h(r)*log(r) integrand is non-negligible.
# Actually h(r)=exp(-r²/(2·400)) — needs r ~ several sigma for cutoff. Cut at r=±200 should suffice. Test increasing further.
for R in [100, 200, 300, 500, 1000]:
 a = mp.quad(arch_integrand, [-R, R]) / (2*mp.pi)
 print(R, a)


100 3.75360575370256606853827526873


200 3.14739710785444358511955338291


300 2.4068380265371241454683920261


500 0.736413822026978247080698052905


1000 -4.06412560513887950337962921486


In [11]:
# The integral is NOT converging — it's diverging. That means my formula is wrong: the "-log π" term needs to be balanced. 
# Actually the formula isn't h(r)·[-log π + Re ψ(1/4+ir/2)] alone — perhaps the integrand should include subtraction of log|r| to be integrable.
# 
# Re ψ(1/4 + ir/2) ~ log(|r|/2) as r→∞. So integrand ~ h(r) log r — this DOES go to 0 if h decays superpolynomially. 
# Gaussian h decays superexponentially, so the integral DOES converge. But for σ=20, h decays only beyond |r|~100.
# 
# The issue is mp.quad's convergence — it's not handling the oscillation. Let me use better quadrature or use the FACT that h is even and Re-ψ is even-ish, and integrate carefully.
# Actually wait: at r=500 with σ=20, h(500) = exp(-500²/800) = exp(-312) ≈ 0. So integrand at 500 is 0.
# Why does extending range change answer? mp.quad must be returning inaccurate values.

# Let me use numpy + scipy.integrate.quad with float precision.
import numpy as np
from scipy.special import digamma as sp_digamma
from scipy.integrate import quad

sigma = 20.0
def arch_int(r):
 return np.exp(-r**2/(2*sigma**2)) * (sp_digamma(0.25 + 1j*r/2).real - np.log(np.pi))

val, err = quad(arch_int, -200, 200, limit=500)
print("scipy quad arch:", val/(2*np.pi), "err:", err)

val2, _ = quad(arch_int, -100, 100, limit=500)
print("100:", val2/(2*np.pi))
val3, _ = quad(arch_int, -300, 300, limit=500)
print("300:", val3/(2*np.pi))


scipy quad arch: 3.9212425018212307 err: 1.7671270400101968e-08
100: 3.921229678571372
300: 3.9212425018212356


In [12]:
# Scipy converges to ~3.9212. The mpmath result was misleading.
# Now check: LHS=5.922, arch=3.921, pole≈2.001, prime_sum negligible.
# RHS ≈ 2.001 + 3.921 - 0 = 5.922. ✓ Matches LHS!

print("LHS (mpmath):", float(lhs))
print("RHS = pole + arch - prime =", 2.0 + 3.9212425 - 0)
print("Diff:", float(lhs) - (2.0 + 3.9212425 - 0))


LHS (mpmath): 5.92186759948766
RHS = pole + arch - prime = 5.9212425
Diff: 0.0006250994876602078


In [13]:
# Excellent — agreement to 6e-4. Discrepancy likely from h(i/2) — recompute with proper pole=2 h(i/2)=2 exp(1/(8σ²))=2 exp(1/3200)≈2.0006250977.
pole_val = 2*np.exp(1/(8*sigma**2))
print("pole exact:", pole_val)
print("RHS:", pole_val + 3.9212425018212356)
print("LHS:", float(lhs))
print("Diff:", float(lhs) - (pole_val + 3.9212425018212356))


pole exact: 2.0006250976664233
RHS: 5.921867599487658
LHS: 5.92186759948766
Diff: 1.7763568394002505e-15


In [14]:
# PERFECT match to machine precision. Formula is verified.
# 
# Summary of confirmed Weil explicit formula (for ζ):
# Σ_γ h(γ) = 2 h(i/2) + (1/(2π)) ∫ h(r) [Re ψ(1/4 + ir/2) - log π] dr - 2 Σ_n (Λ(n)/√n) g(log n)
# 
# where g(x) = (1/(2π)) ∫ h(r) e^{-irx} dr (inverse Fourier transform of h, divided by 2π in our conv).
# 
# Now generalize to general L-functions with completed L-function:
# Λ_L(s) = q^{s/2} ∏_j Γ((s+κ_j)/2) L(s), Λ_L(s) = ε Λ̄_L(1-s).
# 
# - q = conductor.
# - Λ_L(n) = von Mangoldt coefficients defined by -L'/L (s) = Σ Λ_L(n)/n^s.
# - Poles: only if L has poles (e.g., ζ at s=1).
#
# Explicit formula (general):
# Σ_γ h(γ) = (pole contributions: e.g., h(-i/2)+h(i/2) for simple pole at s=1)
# + (log q)/(2π) ∫ h(r) dr 
# + (1/(2π)) ∫ h(r) Σ_j Re ψ((1/2 + κ_j + ir)/2)/?? hmm need to get details.
#
# **Let's derive**: take logarithmic derivative of Λ_L(s)/Λ_L(1) somewhere; standard derivation gives
# For each Γ factor Γ((s+κ_j)/2): contributes (1/2) ψ((s+κ_j)/2) to log derivative.
# Evaluated at s=1/2+ir, real part: (1/2) Re ψ((1/2+κ_j+ir)/2) = (1/2) Re ψ(1/4 + κ_j/2 + ir/2).
# 
# Integrated as 2× (from FE symmetry s↔1-s), the contribution to arch term per Γ-factor is:
# (1/(2π)) ∫ h(r) Re ψ(1/4 + κ_j/2 + ir/2) dr.
# 
# So general formula:
# Σ_γ h(γ) = (poles)
# + (log q)/(2π) ∫ h(r) dr
# + Σ_j (1/(2π)) ∫ h(r) Re ψ(1/4 + κ_j/2 + ir/2) dr
# - 2 Σ_n (Λ_L(n)/√n) g(log n)
# 
# For ζ: q=1, but we have a π^{-s/2} factor which is NOT included in the "q^{s/2} ∏Γ" template. Standard convention is to include π^{-s/2} in archimedean: γ_ℝ(s) = π^{-s/2}Γ(s/2). So Λ_ζ(s) = π^{-s/2} Γ(s/2) ζ(s).
# Log-derivative of π^{-s/2}: -(1/2) log π. So we get -log π once (not twice). That's why we see "−log π" in arch term, matching our formula. ✓
# 
# For Davenport-Heilbronn L_DH: This is not in the Selberg class — it doesn't have an Euler product. But it DOES satisfy a functional equation (Davenport-Heilbronn). It has conductor 5 (?) and gamma factor like ζ.
# Actually L_DH = (1/(2 sin θ))(L(s,χ_1) - e^{iθ} L(s,χ_2)) for some Dirichlet characters mod 5 — it satisfies FE with conductor 5 and appropriate gamma factor.
# 
# **Important**: Since L_DH has no Euler product, Λ_DH(n)/n^s expansion doesn't have nice coefficients via primes — there IS a Dirichlet series -L_DH'/L_DH = Σ a_n/n^s where a_n is NOT supported on prime powers.
# For the test to be a genuine test of GRH, we need to use the CORRECT prime/arithmetic side of the explicit formula for L_DH. If we use the wrong arithmetic side, we won't have a meaningful test.
# 
# Hmm. The hypothesis is that Weil's quadratic form yields negative λ_min for L_DH. The explicit formula for L_DH still holds as an identity (Σ zeros = arch + arithmetic) — it's just that the "arithmetic" side involves coefficients of -L_DH'/L_DH, not Λ(n)/√n.
# 
# This is important: we cannot just plug in Λ(n) for L_DH.
# 
# Let me investigate L_DH structure.
print("Need to investigate L_DH arithmetic side.")


Need to investigate L_DH arithmetic side.


In [15]:
# The classical Davenport-Heilbronn function is defined as
# L_DH(s) = (1/(2*sin(θ)/sqrt(5))) * ... or simpler:
# 
# Let χ be the non-real Dirichlet character mod 5 with χ(2)=i. Then χ takes values χ(1)=1, χ(2)=i, χ(3)=-i, χ(4)=-1, χ(5)=0.
# Define f(s)= L(s,χ) and g(s)=L(s,χ̄). Then DH function:
# L_DH(s) = (1-iξ)/2 · f(s) + (1+iξ)/2 · g(s)
# where ξ = (√10−2√5−2)/(√5−1) ... a specific real number chosen to satisfy the functional equation.
# 
# It has the SAME functional equation as L(χ,s) [conductor 5, gamma factor Γ((s+1)/2) since χ is odd], but it's a real-linear combination so no Euler product.
# 
# The Dirichlet coefficients: a_n = (1-iξ)/2 · χ(n) + (1+iξ)/2 · χ̄(n) = Re[χ(n)] + ξ · Im[χ(n)] (real coefficients).
# So a_1=1, a_2 = ξ·1 = ξ, a_3=-ξ, a_4=-1, a_5=0, a_6=ξ, ... periodic mod 5.
# 
# CRUCIALLY: L_DH has no Euler product. So -L_DH'/L_DH does NOT equal Σ Λ(n) a_n / n^s in any clean way.
# The "Λ_L(n)" coefficients of -L_DH'/L_DH are computed by the Newton identity from a_n coefficients of L_DH itself:
# If L_DH(s) = Σ a_n/n^s, then -L_DH'(s)/L_DH(s) = Σ Λ_DH(n)/n^s where Λ_DH(n) is computed by
# Σ_{d|n} Λ_DH(d) · a_{n/d} = a_n · log n (Wikipedia: this is the convolution relation).
# Hmm actually that's wrong. Let me think.
# 
# Identity: if A(s)=Σa_n/n^s, then -A'(s)/A(s)=Σb_n/n^s where the b_n satisfy: 
# −a_n log n = Σ_{d|n} a_d b_{n/d} (... convolution since (logn)·a_n is coefficient of -A')
# In Dirichlet convolution: -a*log = b * a, so b = (-a*log) * a^{-1}.
# 
# So Λ_DH(n) is supported on ALL n (not just prime powers) in general.
# 
# For positivity test, we use the explicit formula:
# Σ_ρ h(γ_ρ) = arch term - prime/arithmetic sum + pole terms.
# 
# Where for L_DH, "prime sum" is replaced by:
# 2 Σ_{n≥1} Λ_DH(n)/√n · g(log n)
# 
# Note also under GRH-violation, ρ = β + iγ with β ≠ 1/2. The zero-sum side in the explicit formula evaluates h at (ρ-1/2)/i = γ - i(β-1/2). For off-line zeros, h is evaluated at COMPLEX points (γ - i(β-1/2)) and (γ + i(β-1/2)) (the symmetric pair).
# 
# This is where the test detects GRH violation: if h is real on R but the sum over zeros includes off-line zeros, h evaluated at γ ± i(β-1/2) can be different from h evaluated at real γ — and importantly, for h = |g|², we have h(z) = g(z)g(-z) (for real z=t, h(t)=g(t)g(-t)=g(t)² if g even). Complex shifts may make h negative or large.
# 
# OK so the implementation must:
# (a) For ζ, L(χ), L(Δ): use Λ_L(n) supported on prime powers (Euler products). Test should give M ≥ 0 (under GRH).
# (b) For L_DH: use Λ_DH(n) from -L_DH'/L_DH on all n. Include both critical-line zeros AND off-line zeros (each contributing h evaluated at complex z=γ-i(β-1/2)).
# 
# This is computationally feasible but requires care. Let me build it.

# However, for L(Δ,s) (Ramanujan delta): we don't have the zero list. So we cannot include L(Δ,s) in the test.
# The hypothesis asks for all five, but we're missing L(Δ) and ζ_δ zero lists.
# 
# We'll proceed with the four available: ζ, L(χ_5) (real quadratic char), L_DH (with off-line), and note L(Δ,s) and ζ_δ are not available.
print("OK; build matrix for ζ, L(χ_5), L_DH with off-line zeros.")


OK; build matrix for ζ, L(χ_5), L_DH with off-line zeros.


In [16]:
# Now define the test functions. For Weil positivity quadratic form, choose g_j: R→R, even, Schwartz, with
# COMPACTLY SUPPORTED Fourier transform (so prime sum is finite).
# 
# Choose ĝ_j to be even bump functions of compact support on [-X, X]. Then:
# g_j(t) = (1/(2π)) ∫_{-X}^{X} ĝ_j(u) e^{itu} du = (1/π) ∫_0^X ĝ_j(u) cos(tu) du.
# 
# Pick: ĝ_j(u) = u · cos(j π u / X) on [-X,X], or simpler: ĝ_j(u) = (1 - |u|/X) · cos(j π u/X) for j=0,1,2,...
# 
# But then the QUADRATIC FORM is built on h_{jk}(t) = g_j(t) g_k(t).
# Its Fourier transform is (ĝ_j * ĝ_k)/(2π) which is supported on [-2X, 2X].
# So prime sum has support log n ≤ 2X.
# 
# To keep computations manageable: X = 1. Then prime sum has log n ≤ 2, i.e., n ∈ {2, 3, 4, 5, 6, 7}.
# 
# Actually wait. The arithmetic sum involves g(log n) where g = (FT of h)/(2π), so the SUM convergence requires...
# Let me recompute: h_{jk}(t) = g_j(t) g_k(t). 
# Define G_{jk}(x) := (1/(2π)) ∫ h_{jk}(r) e^{-irx} dr = (1/(2π)) (FT of h_{jk})(x).
# FT(g_j · g_k)(x) = (1/(2π)) (ĝ_j ⋆ ĝ_k)(x), so G_{jk}(x) = (1/(4π²)) (ĝ_j ⋆ ĝ_k)(x), supported on [-2X, 2X].
# Hmm too many factors. Let me redo using cleaner conventions.
# 
# Actually, let me drop one normalization: use ĝ defined as ĝ(u)=∫g(t)e^{-itu}dt; inverse: g(t)=(1/(2π))∫ĝ(u)e^{itu}du.
# Then if g_j real even, ĝ_j real even.
# 
# Fact: FT of (g_j(t) g_k(t)) is (1/(2π)) (ĝ_j ⋆ ĝ_k)(u). 
# Verify: g_j(t)g_k(t) = (1/(2π)²) ∫∫ ĝ_j(u)ĝ_k(v) e^{it(u+v)} du dv. Setting w=u+v: 
# = (1/(2π)²) ∫ e^{itw} (∫ ĝ_j(u)ĝ_k(w-u)du) dw = (1/(2π))·(1/(2π))(ĝ_j⋆ĝ_k)(w) integrated... so its FT is (ĝ_j ⋆ ĝ_k)(w)/(2π). ✓
# 
# In our explicit formula:
# Σ h(γ) = poles + (1/(2π))∫ h(r) Ω(r) dr - 2 Σ_n Λ_L(n)/√n · g_L(log n)
# 
# where g_L(x) = (1/(2π)) ∫ h(r) e^{-irx} dr = FT(h)(x)/(2π).
# 
# For h = g_j · g_k: g_L(x) = (1/(2π)) · (ĝ_j ⋆ ĝ_k)(x)/(2π) = (ĝ_j ⋆ ĝ_k)(x)/(4π²).
# 
# That's the arithmetic side. Confusing factors. Let me JUST set up everything numerically and verify by independent computation.

# Simpler approach: Choose g_j(t) directly as a real function with explicit ĝ_j.
# I'll let X (the support half-width of ĝ_j) be a parameter.
# 
# Basis: For j=0,1,...,J-1, let
# ĝ_j(u) = cos(j π u/(2X)) · (1-|u|/X)_+ for j even
# ĝ_j(u) = sin((j+1) π u/(2X)) · sign(u) · (1-|u|/X)_+ for j odd (still even functions since sin·sign is even)
# 
# Hmm. Easier: just use ĝ_j(u) = (1-|u|/X)_+ · cos(j π u/X) for j=0,1,2,... — these are even (cos is even).
# 
# This gives a J-dimensional basis with compactly supported ĝ_j. Then g_j(t) is real-analytic, decays like 1/t².
# 
# For the matrix construction, the cleanest is:
# M_jk = Σ_ρ g_j(γ_ρ - i(β_ρ-1/2)) g_k(γ_ρ + i(β_ρ-1/2)) [pairing both ρ and 1-ρ̄ symmetric]
# − [Arch + Prime + Pole terms for h_jk = g_j g_k]
# 
# Wait — for off-line zeros, each ρ=β+iγ comes paired with 1-ρ̄ = 1-β+iγ (FE symmetry). In our basis where h is REAL on real axis, we need to be careful about how off-line zeros contribute.
# 
# Standard formula: zero side is Σ_ρ h((ρ-1/2)/i) where the sum is over ALL nontrivial zeros (each as (1/2+τ) with τ = γ_ρ + i (β_ρ-1/2) sort of). Actually:
# ρ = 1/2 + iτ_ρ ⇒ τ_ρ = (ρ-1/2)/i.
# Under FE, if ρ is zero so is 1-ρ̄; and 1-ρ̄ = 1/2 - iτ̄_ρ + ... = 1/2 + i(-τ̄_ρ). So τ for 1-ρ̄ is -τ̄_ρ.
# For ρ = β+iγ: τ_ρ = γ - i(β-1/2). For 1-ρ̄ = (1-β)+iγ: τ_{1-ρ̄} = γ + i(β-1/2). They are complex conjugates of each other.
# So h evaluated at τ_ρ + h at τ_{1-ρ̄} = h(τ) + h(τ̄), which is 2 Re h(τ) for h taking complex conjugate values appropriately.
# If h is real-valued on R and analytic, h(τ̄) = conj(h(τ)) (Schwarz reflection), so sum = 2 Re h(τ_ρ).
# 
# In our case h = g_j g_k. For BILINEAR M_jk we have:
# M_jk = Σ_ρ [contribution from {ρ, 1-ρ̄}] of g_j(τ)g_k(τ) — but for bilinear we want Σ g_j(τ_ρ) g_k(τ_ρ) over all ρ (including 1-ρ̄)
# = Σ_{ρ on critical line, β=1/2} 2 g_j(γ_ρ)g_k(γ_ρ) [both ρ and conjugate ρ̄ since γ ↦ -γ same g]
# WAIT: zeros come in (ρ, ρ̄) pairs too since L has real coefficients. So for each γ>0 there's also -γ.
# + Σ_{ρ off line, β≠1/2} g_j(τ_ρ)g_k(τ_ρ) + g_j(-τ̄_ρ)g_k(-τ̄_ρ) + g_j(τ̄_ρ)g_k(τ̄_ρ) + g_j(-τ_ρ)g_k(-τ_ρ)
# 
# (4 zeros: ρ, ρ̄, 1-ρ, 1-ρ̄ in general, for real-coefficient L).
# 
# Since g_j is even and real on R, with even ĝ_j: g_j(z) is even and real-on-real. g_j(-z) = g_j(z); g_j(z̄) = conj(g_j(z)).
# So the four contributions = 2[g_j(τ)g_k(τ) + g_j(τ̄)g_k(τ̄)] = 2·2 Re[g_j(τ)g_k(τ)] = 4 Re[g_j(τ)g_k(τ)].
# 
# Easier: For each zero in the off-line CSV (which lists only β>1/2, γ>0 with implicit partners), the contribution to the zero-sum is 
# 4 Re[g_j(γ - i(β-1/2)) g_k(γ - i(β-1/2))].
# 
# For critical-line zeros (β=1/2): contribution per (γ>0) zero is 2 g_j(γ) g_k(γ) (one for γ, one for -γ).
# 
# Let me proceed.
print("formulation done")


formulation done


In [17]:
# Implementation
# Define test functions with X = support half-width
# 
# ĝ_j(u) = (1 - |u|/X) cos(j π u / X) for |u| ≤ X
# 
# Then g_j(t) = (1/(2π)) ∫_{-X}^X (1-|u|/X) cos(j π u/X) e^{itu} du
# = (1/π) ∫_0^X (1-u/X) cos(j π u/X) cos(tu) du
# 
# Closed form: let a = tX, b = jπ. Then
# I = ∫_0^1 (1-v) cos(b v) cos(a v) dv · X (with u=Xv)
# We can compute symbolically or numerically.

import numpy as np
from scipy.integrate import quad as squad

X = 2.0 # support half-width of ĝ_j; primes contribute for log n ≤ 2X = 4 → n ≤ 54
J = 6 # number of basis functions

def g_j(t, j, X=X):
 # g_j(t) real for real t. Implement complex argument support: t can be complex.
 # Use closed form: g_j(t) = (1/π) ∫_0^X (1-u/X) cos(j π u/X) cos(tu) du
 # For complex t, cos(tu) = (e^{itu}+e^{-itu})/2.
 # Just integrate numerically with mpmath for safety / complex args.
 # For purely real t, use scipy.
 if isinstance(t, complex) or (hasattr(t,'imag') and t.imag != 0):
 # complex integration
 a = complex(t)
 integ_re = squad(lambda u: (1-u/X)*np.cos(j*np.pi*u/X)*np.cos(a.real*u)*np.cosh(a.imag*u), 0, X)[0]
 integ_im = squad(lambda u: (1-u/X)*np.cos(j*np.pi*u/X)*np.sin(a.real*u)*np.sinh(a.imag*u), 0, X)[0]
 return (integ_re + 1j*integ_im)/np.pi
 # Note cos(a*u) for complex a = cos((re+i im)u)=cos(re·u)cosh(im·u) − i sin(re·u)sinh(im·u)
 # But sign: actually cos(a u) = cos(re·u)cos(i·im·u)−sin(re·u)sin(i·im·u) = cos(re·u)cosh(im·u) - i sin(re·u)sinh(im·u). 
 # We want the FULL integral with this complex cosine. So g_j(complex t) = real part comes from cos cosh; imaginary from -sin sinh.
 else:
 integ = squad(lambda u: (1-u/X)*np.cos(j*np.pi*u/X)*np.cos(t*u), 0, X)[0]
 return integ/np.pi

# Quick test
print("g_0(0)=", g_j(0,0)) # should equal (1/π)·∫_0^X (1-u/X) du = (1/π)·X/2
print("expected:", X/(2*np.pi))
print("g_0(1)=", g_j(1,0))
print("g_1(0)=", g_j(0,1))
# expected: (1/π)·∫_0^X (1-u/X) cos(πu/X) du
def expected_g1_0():
 return squad(lambda u: (1-u/X)*np.cos(np.pi*u/X), 0, X)[0]/np.pi
print("expected g_1(0):", expected_g1_0())
print("g_0(1+1j) ←complex:", g_j(1+1j, 0))


g_0(0)= 0.3183098861837907
expected: 0.3183098861837907
g_0(1)= 0.22538676918042805
g_1(0)= 0.12900613773279795
expected g_1(0): 0.12900613773279795
g_0(1+1j) ←complex: (0.2624381324980851+0.2041660169073868j)


In [18]:
# Test that g_j is even and analytic: g_j(-t)=g_j(t), g_j(t̄)=conj(g_j(t))
print(g_j(-1.0, 0), "vs", g_j(1.0, 0)) # should be equal (even)
print(g_j(1-1j, 0), "vs conj of g_j(1+1j,0)=", g_j(1+1j,0).conjugate())
# Good


0.22538676918042805 vs 0.22538676918042805
(0.2624381324980851-0.2041660169073868j) vs conj of g_j(1+1j,0)= (0.2624381324980851-0.2041660169073868j)


In [19]:
# Build the matrix M_jk for each L-function. Components:
#
# M_jk = ZeroSum_jk - ArchSum_jk - PrimeSum_jk - PoleTerm_jk
#
# Where (per derivation):
# - ZeroSum_jk = Σ over ALL zeros ρ of g_j(τ_ρ) g_k(τ_ρ), τ_ρ = (ρ-1/2)/i.
# - ArchSum_jk = (1/(2π)) ∫ g_j(r)g_k(r) Ω_L(r) dr with Ω_L(r) = log q + Σ_l Re ψ(1/4+κ_l/2+ir/2).
# - PrimeSum_jk = 2 Σ_n (Λ_L(n)/√n) G_jk(log n), where G_jk = FT(g_j g_k)/(2π) = (ĝ_j*ĝ_k)/(2π)² = (ĝ_j*ĝ_k)/(4π²).
# - PoleTerm_jk: 2·(g_j(i/2)g_k(i/2)) for ζ (other L's have no pole).
#
# Then in our formulation Σ_ρ ... = Arch + Prime + Pole, so M_jk = ZeroSum - Arch - Prime - Pole should be ≈ 0 (the explicit formula!).
# 
# WAIT — that's the point. Under GRH (all ρ on critical line), the LHS Σ g_j g_k applied to real γ becomes a sum of POSITIVE numbers if j=k (g_j(γ)² ≥ 0). But the bilinear form M_jk ≡ 0 identically because of the explicit formula!
# 
# Hmm so M_jk ≡ 0 identically? Then λ_min = 0 and the test is trivial.
# 
# Oh I see my confusion. The Weil positivity functional W(h) is NOT zero — the explicit formula gives an IDENTITY: 
# Σ_ρ h = pole + arch - prime sum (= pure arithmetic data).
# 
# So if you USE the formula identity, Σ - arch - prime - pole = 0.
# 
# The TEST is: given some prescribed arch + prime + pole (computed from the L-function side), and given a SET OF ZEROS (which may or may not all be on the critical line), the SUM Σ h(γ_ρ over CRITICAL LINE only, with REAL γ) is COMPARED to (pole + arch - prime sum). The discrepancy reveals off-line zeros.
# 
# More precisely Weil positivity:
# Define W(h) := pole + arch - prime - Σ h(γ over real zeros only). Then under GRH (no off-line zeros), the sum over ρ IS the sum over real γ_ρ, so the explicit formula says W(h) = 0. If there ARE off-line zeros, W(h) = -Σ_{off-line ρ} h(τ_ρ).
# 
# For h(t) = |g(t)|² = g(t)·g(-t) — but for our g real even, g(t)g(-t)=g(t)².
# For off-line ρ: τ_ρ = γ - i(β-1/2), complex. h(τ_ρ) = g(τ_ρ)² is complex (or real but not positive).
# 
# This is where things get subtle. Let me re-derive carefully.

# Setup for QUADRATIC FORM:
# Define Weil functional on EVEN Schwartz test functions h:
# W(h) = arch(h) + pole(h) - prime(h) - Σ_{ρ real}( h(γ_ρ) ) 
# (where "ρ real" means we PRESCRIBE the sum is only over critical-line zeros — but we don't know if all zeros are critical!)
# 
# Alternative formulation: Use ALL known zeros (critical + off-line).
# W'(h) = arch(h) + pole(h) - prime(h) - Σ_{ALL ρ} h(τ_ρ)
# 
# By explicit formula identity, W'(h) = 0. So the test of positivity using ALL zeros is trivial.
# 
# The non-trivial Weil positivity test asks: 
# "Use ALL zeros that come from the explicit formula side (the FULL set, including off-line). For h = |g|², we have:
# Σ_{all ρ} h(τ_ρ) = arch + pole - prime."
# 
# Now: for critical-line zeros, h(τ_ρ) = g(γ_ρ)² ≥ 0. 
# For off-line zeros, h(τ_ρ) = g(γ - i(β-1/2))² — this is a COMPLEX number; combined with its conjugate partner gives 2 Re[g(τ)²].
# Under GRH, the sum Σ h(τ_ρ) is a sum of non-negative reals: ≥ 0.
# If there are off-line zeros, the sum may have negative contributions (if Re[g(τ)²] < 0), or the EQUALITY (sum=arch+pole-prime) is maintained but the "zero side" no longer manifestly positive.
# 
# So Weil's criterion says: GRH ⇔ arch + pole - prime ≥ 0 for all h = |g|².
# ⇔ the quadratic form M defined by M_jk = "arch(g_jg_k) + pole - prime(g_jg_k)" is positive semi-definite.
# 
# **The matrix to compute is**:
# M_jk := Arch_jk + Pole_jk − Prime_jk
# 
# (NO zero side — it's defined purely from arithmetic data of L.)
# 
# Then:
# Under GRH (only critical-line zeros), M_jk = Σ_{real ρ} g_j(γ_ρ) g_k(γ_ρ), which is PSD (Gram matrix).
# If off-line zeros exist, M_jk = Σ_{all ρ} g_j(τ_ρ) g_k(τ_ρ), which has contributions from complex τ_ρ that can break PSD.
# 
# THIS is the correct formulation. The matrix M is PSD iff GRH holds.
# 
# OK so we compute M_jk = Arch_jk + Pole_jk - Prime_jk for each L, and check λ_min.
# Note: NOT using zero list at all in M (it's the arithmetic side). But we use zero list as a check.
print("Quadratic form: M = Arch + Pole - Prime (arithmetic side). Test PSD.")


Quadratic form: M = Arch + Pole - Prime (arithmetic side). Test PSD.


In [20]:
# Wait — re-read the research-objective: 
# "...constructs the matrix M_jk = Q(φ_j, φ_k) representing the quadratic form on this basis.
# The zeros ρ used in the sum will be the full set (critical and off-line) for L_DH."
# 
# So the objective DOES want us to use the zero sum side. The intended formula:
# Q(φ) := Σ_ρ φ̂(γ_ρ) - A_∞(φ) - P(φ).
# 
# So Q(φ_j, φ_k) = Σ_ρ φ̂_j(γ_ρ)·φ̂_k(γ_ρ) - arch(φ_j, φ_k) - prime(φ_j, φ_k).
# 
# But the explicit formula identity says this is ZERO (up to pole). So unless we use a TRUNCATED zero set, M is 0.
# 
# Re-reading: "The zeros ρ used in the sum will be the full set (critical and off-line) for L_DH."
# This implies the "full set" is the 5000+110 zeros we have. For ζ, the "full set" is the 5000 known zeros.
# But that's still a TRUNCATION! Only the first N zeros.
# 
# OK now it makes sense. The matrix uses TRUNCATED zero sum:
# M_jk = Σ_{first N zeros} φ̂_j(γ_ρ) φ̂_k(γ_ρ) - A_∞ - P
# 
# For sufficiently small truncation effect (h-test functions concentrating support away from large γ), this should be close to 0.
# 
# Under GRH (all 5000 zeros on line, real γ_ρ), Σ φ̂_j(γ_ρ) φ̂_k(γ_ρ) = sum of products of REAL g_j(γ)g_k(γ), giving Gram-like PSD contribution.
# Then M = (PSD piece) - (arith piece). If truncation captures most of the zeros that matter for h, M ≈ 0 from explicit formula.
# 
# Under non-GRH (L_DH with off-line ρ), the zero sum is now Σ_{critical} + Σ_{off-line} with complex contributions, and the equality may shift M from 0 by amounts revealing off-line zeros' negative contributions.
# 
# Actually I now think the objective phrasing is confusing. The CORRECT Weil-positivity matrix is purely arithmetic:
# M_jk = A_∞(g_j g_k) + Pole(g_j g_k) - P(g_j g_k).
# Then by the explicit formula identity, 
# M_jk = Σ_{ALL ρ in L's spectrum} g_j(τ_ρ) g_k(τ_ρ).
# 
# - For ζ, L(χ), L(Δ): if GRH holds, all τ_ρ are real, M_jk is a Gram-like matrix = Σ g_j(γ) g_k(γ), PSD.
# - For L_DH: τ_ρ has imaginary parts for off-line zeros, so M = Σ g_j(τ)g_k(τ) over all zeros may not be PSD.
# 
# This is the most natural and reproducible test. Let me proceed with BOTH interpretations:
# (A) M = Arith side (Arch + Pole - Prime). Compare λ_min across L. 
# (B) M = Truncated zero side using known N=5000 zeros. Then λ_min ≈ 0 ± truncation error.
# 
# The objective text says (B). Let me do (B) primarily since it's what's asked.
# 
# Actually re-reading once more: "Q(φ) which computes Σ_ρ φ̂(γ_ρ) - A_∞(φ) - P(φ)"
# So Q is the discrepancy. Under GRH and full zero sum, Q=0. With truncated zero sum AND off-line zeros included, Q reflects:
# - truncation error from missing high zeros
# - any structural failure of positivity from off-line zeros
# 
# I'll compute BOTH and report both — but focus on the arithmetic-side PSD test which is the proper "Weil positivity test".

# Let me just compute (A) — the arithmetic-side matrix — and check PSD. This is the rigorous Weil positivity test.
# Then also compute the truncated zero-side bilinear form and show it ≈ M_arith (consistency check / explicit formula numerical verification).
print("Proceeding with arithmetic-side Weil matrix M_jk = Arch + Pole - Prime")


Proceeding with arithmetic-side Weil matrix M_jk = Arch + Pole - Prime


In [21]:
# Actually, the cleanest interpretation reconciling everything:
#
# The OBJECTIVE explicitly says "Q(φ) computes Σ_ρ φ̂(γ_ρ) - A_∞(φ) - P(φ)".
# We have the ZEROS (5000 critical for each, plus 110 off-line for L_DH). The objective treats this as a finite, 
# computable functional Q.
# 
# When we compute Q using the FINITE truncated zero list:
# Q_jk^trunc(φ_j,φ_k) = Σ_{n=1}^{N=5000} φ̂_j(γ_n) φ̂_k(γ_n) + Σ_{off-line ρ} ... - A_∞(φ_j φ_k) - P(φ_j φ_k) - Pole
# 
# Under the explicit formula (which is the IDENTITY for the INFINITE zero sum minus arch minus prime minus pole),
# Σ_{ALL ρ} ... - A_∞ - P - pole = 0
# Hence
# Q_jk^trunc = - Σ_{ρ truncated} φ̂_j(γ_ρ) φ̂_k(γ_ρ) (i.e. the negative of the tail of zero sum)
# 
# So the matrix M is the NEGATIVE TAIL.
# 
# Under GRH (all tail zeros on critical line, γ real, large):
# -tail = -Σ_{n>N} g_j(γ_n) g_k(γ_n) (sum of products of real values, possibly small for low j, large g support)
# 
# The MINIMUM EIGENVALUE of -tail = -MAX eigenvalue of tail Gram. If g_j decay slowly enough that some tail mass remains, λ_min(M) < 0 even under GRH!
# 
# So as posed, the test will give NEGATIVE λ_min for ALL functions (due to truncation), not just L_DH. That's an artifact of truncation.
# 
# **The proper Weil positivity test uses the INFINITE zero sum (or equivalently the arithmetic side)**. The arithmetic side gives an EXACTLY computable quantity:
# M_jk = A_∞(g_j g_k) + Pole - P(g_j g_k)
# This is the true Weil matrix; under GRH it's PSD (because it equals Σ_ALL zeros g_j(γ)g_k(γ)).
# 
# I'll compute this for ζ, L(χ_5), and L_DH (using L_DH's arithmetic side via Dirichlet series).
# For L_DH, the "prime sum" is replaced by an analogous arithmetic sum on Λ_DH(n), which we compute from -L_DH'/L_DH series.
# 
# This is the rigorous, well-defined, finitely-computable test.

# Let me code it up. First the test functions and their products:
# g_j(t) and g_j(t)g_k(t) need to be evaluated for the arch integral (real t) and FT for prime sum.
# 
# Arch term: (1/(2π)) ∫_{-∞}^{∞} g_j(r) g_k(r) Ω_L(r) dr
# 
# Prime term: 2 Σ_n Λ_L(n)/√n · G_jk(log n), where G_jk(x) = FT(g_j g_k)(x)/(2π) = (ĝ_j ⋆ ĝ_k)(x)/(4π²).
# Since ĝ_j is supported in [-X,X], ĝ_j ⋆ ĝ_k is supported in [-2X, 2X].
# So we only need primes with log n ≤ 2X.
# 
# Let me pick X=2 so 2X=4, n ≤ e^4 ≈ 54.

X = 2.0
J = 5

# Compute G_jk(x) = (ĝ_j ⋆ ĝ_k)(x)/(4π²) by direct integration on [-X,X]:
# G_jk(x) = (1/(4π²)) ∫_{-X}^{X} ĝ_j(u) ĝ_k(x-u) du
# ĝ_j(u) = (1-|u|/X) cos(jπu/X) on [-X,X].

def ghat(u, j, X=X):
 if abs(u) > X: return 0.0
 return (1 - abs(u)/X) * np.cos(j*np.pi*u/X)

def G_jk(x, j, k, X=X):
 # convolution ∫ ĝ_j(u) ĝ_k(x-u) du, supported when |x| ≤ 2X
 if abs(x) > 2*X: return 0.0
 # ĝ_j supported on [-X,X], ĝ_k(x-u) supported on [x-X, x+X]
 lo = max(-X, x-X)
 hi = min(X, x+X)
 if lo >= hi: return 0.0
 val = squad(lambda u: ghat(u,j) * ghat(x-u,k), lo, hi, limit=200)[0]
 return val / (4*np.pi**2)

print("G_00(0)=", G_jk(0,0,0), "G_00(1)=", G_jk(1,0,0), "G_00(5)=", G_jk(5,0,0))
print("G_11(0)=", G_jk(0,1,1))
print("G_01(1)=", G_jk(1,0,1))


G_00(0)= 0.033773727880779265 G_00(1)= 0.024274866914310092 G_00(5)= 0.0
G_11(0)= 0.019453359504060713
G_01(1)= 0.010967250333629788


In [22]:
# Sanity check: verify that G_jk(x) is FT(g_j g_k)/(2π).
# Compute (1/(2π)) ∫ g_j(r) g_k(r) e^{-irx} dr for some x, compare.
def gjgk_FT(x, j, k):
 re = squad(lambda r: g_j(r,j) * g_j(r,k) * np.cos(r*x), -50, 50, limit=300)[0]
 return re/(2*np.pi)

for jj,kk,xx in [(0,0,0), (0,0,1), (1,1,0), (0,1,1)]:
 print(f"j={jj} k={kk} x={xx}: G={G_jk(xx,jj,kk):.8f} vs FT={gjgk_FT(xx,jj,kk):.8f}")


j=0 k=0 x=0: G=0.03377373 vs FT=0.03377370


j=0 k=0 x=1: G=0.02427487 vs FT=0.02427487
j=1 k=1 x=0: G=0.01945336 vs FT=0.01945333


j=0 k=1 x=1: G=0.01096725 vs FT=0.01096725


In [23]:
# Good, agreement confirmed. Now let's set up arch terms for each L-function.
# 
# ζ: Ω_ζ(r) = -log π + Re ψ(1/4 + ir/2)
# L(χ_5) real quadratic, mod 5, EVEN character (since [0,1,-1,-1,1] is even — χ(-1)=χ(4)=1):
# gamma factor Γ(s/2), conductor 5. So Ω = log 5 + Re ψ(1/4 + ir/2). Same digamma as ζ (κ=0). 
# Wait, also need to absorb π^{-s/2}? Standard convention for L(χ) even: Λ(s,χ) = (5/π)^{s/2} Γ(s/2) L(s,χ).
# So we get log(5/π) = log 5 - log π in archimedean (just like ζ formula but with +log 5 added).
# L_DH: Same FE as L(χ_4 mod 5), but χ_4 is the order-4 (odd) character. For odd χ: Λ(s,χ) = (q/π)^{(s+1)/2} Γ((s+1)/2) L(s,χ).
# gamma factor Γ((s+1)/2), so κ=1. Conductor q=5. Archimedean: log(5/π) [-log π absorbed] + Re ψ((1/2+1+ir)/2) = Re ψ(3/4 + ir/2).
# Specifically: completed L_DH(s) = (5/π)^{(s+1)/2} Γ((s+1)/2) L_DH(s). 
# Ω_DH(r) = log 5 - log π + Re ψ(3/4 + ir/2).
# 
# Wait — for even χ (which χ_5=[0,1,-1,-1,1] is, since χ(-1)=χ(4)=1): Λ(s) = (q/π)^{s/2} Γ(s/2) L(s) → Ω = log q - log π + Re ψ(1/4+ir/2)
# For odd χ: Λ(s) = (q/π)^{(s+1)/2} Γ((s+1)/2) L(s) → Ω = log q - log π + Re ψ(3/4 + ir/2)
# 
# Let me verify by re-deriving from a specific FE.
# 
# DEFINITION: For Λ(s) = A^s ∏_j Γ((s+κ_j)/2) L(s) with FE Λ(s)=ε Λ̄(1-s),
# log derivative of Λ: Λ'/Λ = log A + (1/2) Σ_j ψ((s+κ_j)/2) + L'/L.
# 
# Standard explicit formula derivation with even smooth h:
# Σ_ρ h(γ) = 2 log A · (1/(2π))∫h(r)dr + Σ_j (1/(2π))∫h(r) Re ψ((1/2+κ_j+ir)/2) dr - 2 Σ_n Λ_L(n)/√n g(log n) + poles
# 
# (Factor of 2 in front of log A because of pairing s and 1-s.)
# 
# Hmm, let me just empirically determine the coefficients by validating against ζ and L(χ_5).

# For ζ with A = π^{-1/2}, κ_1=0: 
# 2 log A = -log π. log q = 0. OK.
# gamma contribution: (1/(2π))∫ h(r) Re ψ(1/4+ir/2) dr.
# Yes, total Ω_ζ(r) = -log π + Re ψ(1/4 + ir/2). ✓ (matches our verified formula)
#
# For L(χ_5) (even Dirichlet): A = (5/π)^{1/2}, κ_1=0.
# 2 log A = log(5/π) = log 5 - log π.
# Ω(r) = log 5 - log π + Re ψ(1/4 + ir/2).
# 
# For L(χ_4 mod 5) (odd Dirichlet): A=(5/π)^{1/2}, κ_1=1.
# 2 log A = log 5 - log π.
# Ω(r) = log 5 - log π + Re ψ((1/2+1+ir)/2) = log 5 - log π + Re ψ(3/4 + ir/2).
# 
# For L_DH (linear combination of L(χ_4) and L(χ̄_4) with same gamma factor):
# Ω_DH(r) = log 5 - log π + Re ψ(3/4 + ir/2) (same as L(χ_4))
# Conductor = 5, odd gamma factor.

# Let me verify formula numerically for L(χ_5) using LHS Σ over our 5000 zeros.
sigma = 20.0
def h_gauss(r):
 return np.exp(-r**2/(2*sigma**2))

lhs_lchi = 2 * np.sum(h_gauss(lchi_g))
print("LHS lchi:", lhs_lchi)

# Arch for L(χ_5)
def arch_lchi(r):
 return h_gauss(r) * (sp_digamma(0.25 + 1j*r/2).real - np.log(np.pi) + np.log(5))
arch_l = squad(arch_lchi, -200, 200, limit=500)[0]/(2*np.pi)
print("Arch lchi:", arch_l)

# Prime sum for L(χ_5): Λ_L(n) = Λ(n)·χ(n) (only on prime powers since Euler product) — for the real quadratic char [0,1,-1,-1,1]
# Actually for any Dirichlet character, L(s,χ) = ∏_p (1 - χ(p) p^{-s})^{-1}, so 
# -L'/L = Σ_n Λ(n) χ(n)/n^s where Λ(n) is usual von Mangoldt.
# χ(p^k) = χ(p)^k.
chi5 = {1:1, 2:-1, 3:-1, 4:1, 5:0} # χ(n) = legendre(n,5), real quadratic char mod 5
def chi5_val(n):
 return chi5[n % 5] if n % 5 != 0 else 0

def g_gauss(x):
 return (sigma/np.sqrt(2*np.pi))*np.exp(-(sigma*x)**2/2)

from sympy import factorint
prime_lchi = 0
for n in range(2, 50000):
 fac = factorint(n)
 if len(fac)==1:
 p, k = next(iter(fac.items()))
 chiv = chi5_val(p)**k
 if chiv != 0:
 prime_lchi += np.log(p)*chiv/np.sqrt(n) * g_gauss(np.log(n))
print("Prime lchi (raw sum, factor 2 outside):", prime_lchi)
print("2x:", 2*prime_lchi)

# No pole (L(χ_5) has no pole). RHS = Arch - 2*Prime
rhs_lchi = arch_l - 2*prime_lchi
print("RHS lchi:", rhs_lchi)
print("LHS-RHS:", lhs_lchi - rhs_lchi)


LHS lchi: 16.762699120840864
Arch lchi: 16.762699120840857
Prime lchi (raw sum, factor 2 outside): -7.254909061039741e-42
2x: -1.4509818122079482e-41
RHS lchi: 16.762699120840857
LHS-RHS: 7.105427357601002e-15


In [24]:
# Excellent — formula verified to 7e-15 for L(χ_5). Now let me verify for L_DH.
# 
# L_DH Dirichlet coefficients (with our chosen ξ): a_n is real, periodic mod 5.
# Need to determine ξ. The standard Davenport-Heilbronn formula:
# 
# Let χ be the character mod 5 with χ(2)=i. So χ(1)=1, χ(2)=i, χ(3)=-i, χ(4)=-1, χ(5)=0. This is order 4.
# f(s) = L(s, χ), satisfies FE: (5/π)^{(s+1)/2} Γ((s+1)/2) f(s) = ε · (5/π)^{(1-s+1)/2} Γ((1-s+1)/2) f̄(1-s)
# where f̄(s) = L(s, χ̄) and ε is some root number.
# 
# DH function: L_DH(s) = (1/(2 sin θ)) [(e^{iθ}) f(s) + (e^{-iθ}) f̄(s)]·??? 
# 
# Actually I recall L_DH = (1/(2√(1-α²))) ((1+iα) f(s) + (1-iα) f̄(s)) for some real α giving FE.
# The classical choice that makes Λ_DH satisfy FE with conductor 5 is α = tan(θ) where θ relates to ε.
# 
# Alternative & simpler: Davenport-Heilbronn (1936) used a specific α to make the FE have the same form as a Dirichlet L of conductor 5.
# 
# Numerical value: α = tan(θ) where θ = (1/2) arg(...). Often quoted as α = (√10−2√5−2)/(√5−1).
# Let me compute: 
import numpy as np
alpha = (np.sqrt(10-2*np.sqrt(5)) - 2)/(np.sqrt(5)-1)
print("alpha=", alpha)
# Then a_n (real Dirichlet coefficients of L_DH):
# L_DH(s) = (1/(2 sqrt(1+...))) ... let me just verify by computing L_DH at our claimed off-line zeros.


alpha= 0.28407904384041227


In [25]:
# Standard formula: L_DH(s) = (1 - i α)/2 · L(s, χ) + (1 + i α)/2 · L(s, χ̄)
# Equivalently: L_DH(s) = Re[L(s,χ)] + α · Im[L(s,χ)] (since L(s,χ̄) = conj(L(s,χ)) for real s).
# 
# Coefficients: a_n = Re[χ(n)] + α · Im[χ(n)] where χ(n)∈{1,i,-i,-1,0}.
# χ(1)=1: a_1=1
# χ(2)=i: a_2=α
# χ(3)=-i: a_3=-α
# χ(4)=-1: a_4=-1
# χ(5)=0: a_5=0
# And a_n is periodic mod 5: a_{n+5}=a_n. So a is a "fake character mod 5".
# 
# Compute L_DH at z=1/2 + i*5 (just a check) and at one of the off-line zeros.
import mpmath as mp
mp.mp.dps = 50
alpha_mp = (mp.sqrt(10-2*mp.sqrt(5))-2)/(mp.sqrt(5)-1)
print("alpha mp:", alpha_mp)

def L_chi(s, terms=200000):
 # Hurwitz approach? Easier: just sum L(s,χ) = Σ χ(n)/n^s for Re(s)>1; for Re(s)<1, use functional equation via Hurwitz zeta.
 # Use mpmath's dirichlet with character values.
 # χ mod 5: χ(1)=1, χ(2)=i, χ(3)=-i, χ(4)=-1.
 vals = [0, mp.mpc(1,0), mp.mpc(0,1), mp.mpc(0,-1), mp.mpc(-1,0)]
 return mp.dirichlet(s, vals) # mpmath has L(s,χ) via dirichlet([0,χ(1),..,χ(q-1)]) — actually dirichlet(s, [0]+χ_values..., ...)
# 
# mpmath.dirichlet(s, [chi(0),...,chi(q-1)]) for Dirichlet series mod q.

def L_DH(s):
 Lc = L_chi(s)
 Lcbar = L_chi(s).conjugate() if s.imag == 0 else mp.dirichlet(s, [0, mp.mpc(1,0), mp.mpc(0,-1), mp.mpc(0,1), mp.mpc(-1,0)])
 return (1 - 1j*alpha_mp)/2 * Lc + (1 + 1j*alpha_mp)/2 * Lcbar

# Test at off-line zero (0.650786, 114.163343)
s_test = mp.mpc('0.650786', '114.163343')
print("L_DH at off-line zero:", L_DH(s_test))
print("|L_DH|:", abs(L_DH(s_test)))


alpha mp: 0.28407904384041229602829183239312616909108808844573
L_DH at off-line zero: (-0.00003833486276169535210448979715373431752655090607761 - 0.000012539720056741226204266915787481326085761776773571j)
|L_DH|: 0.000040333686690649234270993768542582856541451523570632


In [26]:
# Hmm, |L_DH(0.650786, 114.163343)| ≈ 4e-5, not the 8e-51 expected from the CSV. So my α or formula is slightly off — or the off-line zero requires more precise sigma/t.
# But the CSV says |L_DH|<1e-40 at sigma=0.650830 (not 0.650786 from the canonical R2 list).
# 
# Let me check at the more precise coordinate from the CSV.
s_test2 = mp.mpc(offline.iloc[0]['sigma_str'], offline.iloc[0]['t_str'])
print("at CSV sigma_str, t_str:", s_test2)
print("L_DH:", L_DH(s_test2))
print("|L_DH|:", abs(L_DH(s_test2)))


at CSV sigma_str, t_str: (0.65083008060973701613960429313010536134243011474609 + 114.1633427307569803588194190524518489837646484375j)
L_DH: (1.0051692334550844474082481905829524209862104002668e-16 - 4.9252486705674322393105791571795590250252993486639e-16j)
|L_DH|: 5.0267722899302829593049339718130645409573810350687e-16


In [27]:
# Excellent! |L_DH| ≈ 5e-16 at the CSV coordinate, confirming it IS a zero. The CSV's claim of |L_DH|<1e-40 was at higher precision; my sigma/t are only stored to float precision, so 5e-16 is consistent.
# So my L_DH formula is CORRECT. ✓
# 
# Now verify the explicit formula numerically for L_DH using h=Gaussian. 
# Key wrinkle: the explicit formula's "prime sum" actually has Λ_L from -L'/L = Σ Λ_L(n)/n^s.
# For L_DH which has no Euler product, -L_DH'/L_DH is computed by formal Dirichlet series inversion:
# L_DH(s) = Σ a_n/n^s, -L_DH'(s) = Σ a_n log(n) /n^s
# -L_DH'/L_DH(s) = Σ Λ_DH(n)/n^s where (in Dirichlet convolution): -L'·L^{-1} = Λ_DH * 1_n,
# equivalently a * Λ_DH = a · log (where (a·log)_n = a_n log n).
# 
# So: a_n log n = Σ_{d|n} a_d Λ_DH(n/d).
# Solving: Λ_DH(n) = a_n log n - Σ_{d|n, d<n} Λ_DH(n/d)·a_d / a_1, but a_1=1, so:
# Λ_DH(1) = 0
# Λ_DH(n) = a_n log n - Σ_{d|n, 1<d<n} Λ_DH(n/d) · a_d - Λ_DH(n)·a_1 (last term...wait)
# 
# Let me re-derive. Let A(s) = Σa_n/n^s, define B(s) = -A'/A. Then A·B = -A'. 
# Coefficients: (a*b)_n = -a_n log n ⟹ Σ_{d|n} a_d b_{n/d} = -a_n log n. 
# Wait that gives Λ_L(n) = -(coefficient of n^{-s} in L'/L).
# Standard convention: -L'(s)/L(s) = Σ Λ_L(n)/n^s. So with this convention: A · B = -A' gives Σ_{d|n} a_d b_{n/d} = -a_n log n, so b_n = -(a_n log n + Σ_{1<d<n,d|n} a_d b_{n/d}·...)/a_1 — sign! Let me be careful.
# 
# Define B(s) = -A'(s)/A(s) = Σ b_n/n^s where b_n = Λ_L(n).
# Then A · B = -A'. Coefficients: (Σa_d/d^s)(Σb_m/m^s) = Σ_n (Σ_{dm=n} a_d b_m)/n^s.
# -A'(s) = Σ a_n log n / n^s.
# So Σ_{dm=n} a_d b_m = a_n log n.
# 
# That is: a_n log n = Σ_{d|n} a_d Λ_L(n/d).
# For Euler-product L (like ζ, L(χ)), this reduces to Λ_L(n) = (usual Λ)(n)·χ(n).
# For L_DH (no Euler product), Λ_DH lives on all n.
# 
# Verify: For Λ(1) = ? Take n=1: a_1 log 1 = Σ_{d|1} a_d Λ_L(1/d) = a_1 Λ_L(1), so 0 = Λ_L(1) ⟹ Λ_L(1)=0. ✓
# For n=2: a_2 log 2 = a_1 Λ_L(2) + a_2 Λ_L(1) = Λ_L(2). So Λ_L(2) = a_2 log 2 = α log 2.
# For ζ: a_n = 1 for all n, so Λ_ζ(2) = log 2 ✓.
# 
# Recursion: Λ_L(n) = a_n log n - Σ_{1<d|n} a_d Λ_L(n/d).
# 
# Now build Λ_DH for n up to ~50 (since 2X=4, log n ≤ 4 ⟹ n ≤ 54).

X = 2.0
N_max = 60 # primes/composites up to n=60

a_DH = [0.0] * (N_max+1)
# a_n periodic mod 5 with [a_0=0, a_1=1, a_2=α, a_3=-α, a_4=-1] (a_5=0, repeats)
alpha_f = float(alpha_mp)
for n in range(1, N_max+1):
 r = n % 5
 if r==0: a_DH[n]=0.0
 elif r==1: a_DH[n]=1.0
 elif r==2: a_DH[n]=alpha_f
 elif r==3: a_DH[n]=-alpha_f
 elif r==4: a_DH[n]=-1.0

Lambda_DH = [0.0] * (N_max+1)
Lambda_DH[1] = 0.0
def divisors(n):
 divs = [d for d in range(2, n+1) if n % d == 0]
 return divs
for n in range(2, N_max+1):
 val = a_DH[n] * np.log(n)
 for d in range(2, n+1):
 if n % d == 0 and d < n:
 val -= a_DH[d] * Lambda_DH[n//d]
 # n=n term: d=n, n/d=1, Λ_DH(1)=0 — contributes 0
 Lambda_DH[n] = val

print("Λ_DH for n=2..20:", [round(Lambda_DH[i],6) for i in range(2,21)])
print("Λ_DH(2) expected α log 2 =", alpha_f*np.log(2))
print("Λ_DH(4) =", Lambda_DH[4], "expected? a_4 log 4 - a_2 Λ_DH(2) = -log4 - α·α·log2 =", -np.log(4)-alpha_f**2*np.log(2))


Λ_DH for n=2..20: [np.float64(0.196909), np.float64(-0.312093), np.float64(-1.442232), np.float64(0.0), np.float64(1.936356), np.float64(0.552792), np.float64(0.015891), np.float64(-2.285884), np.float64(0.0), np.float64(2.397895), np.float64(-0.762877), np.float64(-0.728648), np.float64(-2.852032), np.float64(0.0), np.float64(1.38178), np.float64(0.804857), np.float64(0.887357), np.float64(-2.944439), np.float64(0.0)]
Λ_DH(2) expected α log 2 = 0.19690858829414687
Λ_DH(4) = -1.4422319646064572 expected? a_4 log 4 - a_2 Λ_DH(2) = -log4 - α·α·log2 = -1.4422319646064572


In [28]:
# ✓ Recursion verified.
# 
# Verify explicit formula for L_DH using the Gaussian h:
# LHS = Σ over zeros (both critical-line + off-line) of h(τ_ρ).
# 
# Each off-line ρ contributes 4 partners (ρ, ρ̄, 1-ρ, 1-ρ̄), in our convention with even h on real axis: 
# 2 × [h(γ + i(1/2-β)) + h(γ - i(1/2-β))] (also × 2 for ±γ since h is even, but the formula counts both γ and -γ as separate zeros).
# 
# Actually the explicit formula sums over ALL nontrivial zeros (with multiplicity), which for real L includes both ρ and ρ̄. We have stored only γ>0 critical-line zeros (each represents 2 zeros: ±iγ), and off-line zeros with σ>1/2, t>0 (each represents 4 zeros: ρ, ρ̄, 1-ρ, 1-ρ̄).
#
# In our explicit formula Σ_ρ h(γ_ρ) where γ_ρ = (ρ-1/2)/i:
# critical-line: each stored γ>0 means ρ=1/2+iγ and ρ̄=1/2-iγ, contributing h(γ)+h(-γ)=2h(γ) (h even).
# off-line: each stored (σ,t) with σ>1/2, t>0 represents 4 zeros: 
# 1/2+iτ where τ ∈ {t-i(σ-1/2), -t-i(σ-1/2), t+i(σ-1/2), -t+i(σ-1/2)}.
# Since h is even: h(τ)=h(-τ). Among the 4 we get 2 distinct values: h(t-i(σ-1/2)) (appears twice) and h(t+i(σ-1/2)) (appears twice).
# So contribution = 2 h(t-i(σ-1/2)) + 2 h(t+i(σ-1/2)) = 4 Re h(t-i(σ-1/2)) [if h takes conjugate values: h(τ̄)=conj h(τ)].
# 
# For our Gaussian h(z)=e^{-z²/(2σ²)}, h(z̄)=conj h(z), so above gives 4·Re[e^{-(t-i(σ-1/2))²/(2σ²)}].

sigma_g = 20.0
def h_gauss_z(z):
 return np.exp(-z**2/(2*sigma_g**2))

# Critical-line contribution:
lhs_critical = 2*np.sum(h_gauss_z(ldh_g))

# Off-line contribution:
lhs_offline = 0.0
for _, row in offline.iterrows():
 sig = row['sigma']; tt = row['t']
 z = tt - 1j*(sig-0.5)
 lhs_offline += 4 * np.real(h_gauss_z(z))

print("Critical:", lhs_critical, " Off-line:", lhs_offline)
lhs_DH = lhs_critical + lhs_offline
print("LHS DH:", lhs_DH)

# Arch for L_DH: Ω(r) = log 5 - log π + Re ψ(3/4 + ir/2).
from scipy.special import digamma as sp_digamma
def arch_DH(r):
 return h_gauss_z(r) * (sp_digamma(0.75 + 1j*r/2).real + np.log(5) - np.log(np.pi))
arch_DH_val = squad(arch_DH, -200, 200, limit=500)[0]/(2*np.pi)
print("Arch DH:", arch_DH_val)

# Prime sum for L_DH:
# 2 Σ_n Λ_DH(n)/√n · g(log n), with g(x)=(sigma/√(2π))exp(-(σx)²/2)
prime_DH = 0
for n in range(2, N_max+1):
 prime_DH += Lambda_DH[n]/np.sqrt(n) * (sigma_g/np.sqrt(2*np.pi))*np.exp(-(sigma_g*np.log(n))**2/2)
print("Prime DH (2× outside):", 2*prime_DH)

# No pole for L_DH (it has no pole — only ζ has one). RHS = Arch - 2*Prime
rhs_DH = arch_DH_val - 2*prime_DH
print("RHS DH:", rhs_DH)
print("LHS - RHS:", lhs_DH - rhs_DH)


Critical: 17.262131380640493 Off-line: 3.359881051496222e-07
LHS DH: 17.2621317166286
Arch DH: 17.262542992756323
Prime DH (2× outside): 4.121935258418625e-42
RHS DH: 17.262542992756323
LHS - RHS: -0.0004112761277248467


In [29]:
# Small discrepancy (~4e-4). Probably truncation error: missing zeros for |γ|>~5000 (h(5000)=exp(-5000²/800)≈ negligible),
# OR truncation in N_max for Λ_DH. Recall N_max=60: g_gauss(log(60))=σ/√(2π) exp(-(20·4.09)²/2)≈vanishingly small ⟹ prime sum truncation ok.
# 
# More likely: σ=20 may be too narrow on the SHARP DH zero density at small γ. Or arch integral.
# Let me check: maybe my off-line zeros also need their conjugates etc, or maybe I'm double-counting.
# 
# Actually wait — for the 4-zero set {ρ, ρ̄, 1-ρ, 1-ρ̄} of an off-line zero (σ,t) with σ>1/2:
# I claimed contribution = 4 Re h(t-i(σ-1/2)).
# But h_gauss(20)(t-i·0.15) for t=114: 
import numpy as np
z = 114.16 - 0.15j
h = np.exp(-z**2/(2*400))
print("h at off-line z:", h, "Re:", h.real)
# This is tiny, exp(-(114²-0.0225)/800)=exp(-16.3)≈8e-8. So 4×Re ≈ 3.4e-7. Matches.
# So off-line contribution is tiny. The discrepancy 4e-4 isn't from off-line.
# 
# Let me investigate why ζ matched perfectly but DH has 4e-4 error.
# Perhaps my Λ_DH recursion has a sign or coefficient issue. Let me verify directly:
# Compute L_DH'(s)/L_DH(s) numerically at s=2 and compare with Σ Λ_DH(n)/n^2.

import mpmath as mp
mp.mp.dps = 30
# Numerical derivative
s0 = mp.mpf(2)
eps = mp.mpf('1e-20')
# Recompute L_DH with mpmath
def L_chi_mp(s):
 vals = [mp.mpc(0,0), mp.mpc(1,0), mp.mpc(0,1), mp.mpc(0,-1), mp.mpc(-1,0)]
 return mp.dirichlet(s, vals)
def L_chi_bar_mp(s):
 vals = [mp.mpc(0,0), mp.mpc(1,0), mp.mpc(0,-1), mp.mpc(0,1), mp.mpc(-1,0)]
 return mp.dirichlet(s, vals)
def L_DH_mp(s):
 return (1-1j*alpha_mp)/2 * L_chi_mp(s) + (1+1j*alpha_mp)/2 * L_chi_bar_mp(s)

L_val = L_DH_mp(s0)
Lprime = (L_DH_mp(s0+eps) - L_DH_mp(s0-eps))/(2*eps)
print("L_DH(2):", L_val)
print("-L'/L at s=2:", -Lprime/L_val)

# Compare with Σ Λ_DH(n)/n^2 for n=2..N_max:
big_N = 200
a_DH_big = [0.0]*(big_N+1)
for n in range(1, big_N+1):
 r = n%5
 a_DH_big[n] = {0:0.0, 1:1.0, 2:alpha_f, 3:-alpha_f, 4:-1.0}[r]
Lambda_DH_big = [0.0]*(big_N+1)
for n in range(2, big_N+1):
 val = a_DH_big[n]*np.log(n)
 for d in range(2,n+1):
 if n%d==0 and d<n:
 val -= a_DH_big[d]*Lambda_DH_big[n//d]
 Lambda_DH_big[n]=val
s_approx = sum(Lambda_DH_big[n]/n**2 for n in range(2,big_N+1))
print("Σ Λ_DH/n^2 (N=200):", s_approx)


h at off-line z: (8.40780331641947e-08+3.601581069819611e-09j) Re: 8.40780331641947e-08
L_DH(2): (1.00006833780978267565188794187 + 0.0j)
-L'/L at s=2: (-0.0326436174140271031275406540174 + 0.0j)
Σ Λ_DH/n^2 (N=200): -0.03311215807154753


In [30]:
# Close. The numerical -L'/L=-0.0326, vs partial sum -0.0331 (truncation at N=200).
# At larger N, -L'/L is computed exactly by Dirichlet inversion. Good — coefficients are correct.
# 
# Why the 4e-4 mismatch in the explicit formula? Let me check arch integral more carefully.
# Maybe digamma in arch term needs CAREFUL r=0 behavior, but more likely my "Re ψ(3/4+ir/2)" formula is incomplete:
# For odd L, formula has (1/2) ψ((s+1)/2). At s=1/2+ir: (1/2) ψ(3/4 + ir/2). Integration over r picks up factor of 2 from FE symmetry s↔1-s, giving Re ψ(3/4+ir/2). OK.
# 
# Let me increase truncation to be sure:
import numpy as np
big_N2 = 1000
a_DH_big2 = [0.0]*(big_N2+1)
for n in range(1, big_N2+1):
 r = n%5
 a_DH_big2[n] = {0:0.0, 1:1.0, 2:alpha_f, 3:-alpha_f, 4:-1.0}[r]
Lambda_DH_big2 = [0.0]*(big_N2+1)
for n in range(2, big_N2+1):
 val = a_DH_big2[n]*np.log(n)
 for d in range(2,n+1):
 if n%d==0 and d<n:
 val -= a_DH_big2[d]*Lambda_DH_big2[n//d]
 Lambda_DH_big2[n]=val

# Re-do prime sum for N=1000
sigma_g = 20.0
prime_DH_big = 0.0
for n in range(2, big_N2+1):
 prime_DH_big += Lambda_DH_big2[n]/np.sqrt(n) * (sigma_g/np.sqrt(2*np.pi))*np.exp(-(sigma_g*np.log(n))**2/2)
print("Prime DH (N=1000, ×2):", 2*prime_DH_big)
# This contribution remains tiny because of exp(-σ²·log²n/2).

# Check the LHS more carefully. Are we missing zeros? We have 5000 critical zeros up to γ=4981. Beyond that, contribution exponentially tiny. OK.
# 
# Maybe arch integration needs higher precision. Use mpmath:
from mpmath import digamma as mdigamma, quad as mquad, pi as mpi, log as mlog, exp as mexp
mp.mp.dps = 30
def arch_DH_mp(r):
 rr = mp.mpf(r) if not isinstance(r, mp.mpc) else r
 return mp.exp(-rr**2/(2*sigma_g**2)) * (mp.re(mdigamma(mp.mpf('0.75') + 1j*rr/2)) + mlog(5) - mlog(mpi))
# Integrate using mpmath 
val_mp = mp.quad(arch_DH_mp, [-200, 200])/(2*mpi)
print("Arch DH mpmath:", val_mp)
print("Arch DH scipy:", arch_DH_val)


Prime DH (N=1000, ×2): 4.121935258418625e-42


Arch DH mpmath: 17.2182225400399132425664551205
Arch DH scipy: 17.262542992756323


In [31]:
# mpmath says arch = 17.218 vs scipy 17.262 (and LHS critical = 17.262). 
# scipy matches LHS, mpmath disagrees. mpmath quad may not have converged. Let me check several integration ranges:
for R in [50, 100, 200, 400]:
 v = mp.quad(arch_DH_mp, [-R, R])/(2*mpi)
 print(R, v)

# Then for scipy:
from scipy.integrate import quad as squad
for R in [50, 100, 200, 400]:
 def f(r): return np.exp(-r**2/(2*sigma_g**2))*(sp_digamma(0.75+1j*r/2).real + np.log(5)-np.log(np.pi))
 v, e = squad(f, -R, R, limit=500)
 print("scipy", R, v/(2*np.pi), "err", e)


50 16.8860044133242930307419566216


100 17.26159536746593900314465932


200 17.2182225400399132425664551205


400 16.8927235227555628443926969373
scipy 50 16.88600484294303 err 8.497384760281358e-11
scipy 100 17.262522807459014 err 8.498450574384998e-11
scipy 200 17.262542992756323 err 1.3995919720960893e-10
scipy 400 17.262542992756323 err 1.3993077550017853e-10


In [32]:
# Scipy converges to 17.262543. mpmath quad clearly fails to converge.
# LHS = 17.262131. scipy Arch = 17.262543. Diff = -0.000412.
# This is the LHS - RHS = -0.000412.
# 
# This is from TRUNCATION of zero sum. The zeros at γ ~ 100-200 contribute significantly since σ=20.
# Actually no — σ=20 means h decays at scale 20. So zeros γ < ~80 contribute meaningfully.
# We have 5000 zeros, up to γ=4981. So no truncation.
# 
# Then where's 4e-4 coming from? Let me redo with stored zeros at higher precision.

# Use mpmath floats from string zeros directly:
mp.mp.dps = 30
ldh_g_mp = [mp.mpf(s) for s in ldh]
def h_mp(z):
 return mp.exp(-z**2/(2*sigma_g**2))
s_crit = mp.mpf(0)
for g in ldh_g_mp:
 s_crit += h_mp(g)
print("Σ critical (mpmath, just positive γ, then 2×):", 2*s_crit)
# 17.262131 again. 

# Hmm. Looking more carefully — the discrepancy could be from missing low-height zeros for L_DH.
# DH has zeros at γ ≈ 5.094, 8.5, etc. Are all small ones in our list?
print("First 10 L_DH zeros:", ldh_g[:10])
# Wikipedia/literature: L_DH has zeros at γ ≈ 5.094, ... Our first is 5.094. OK.
# Maybe DH also has trivial zeros that affect formula? Trivial zeros of L_DH from gamma factor: at negative integers congruent to -1 mod 2 (for odd char gamma factor Γ((s+1)/2)): trivial zeros at s = -1, -3, -5, ... 
# These are usually excluded from "non-trivial" zero list.
# But the EXPLICIT FORMULA sums over NONTRIVIAL zeros — that should be fine.
# 
# Could be the formula needs a different coefficient. Let me check by adjusting log(5) vs log(5)·(some factor).
# Actually the FE has Λ(s) = (5/π)^{(s+1)/2} Γ((s+1)/2) L(s,χ). For L_DH which is essentially the same shape:
# Λ_DH(s) = (5/π)^{(s+1)/2} Γ((s+1)/2) L_DH(s)
# So A^s = (5/π)^{(s+1)/2}. We need 2 log A in formula derivation. Let me re-derive precisely.
# 
# Setup: completed Λ_L(s) = A^s · γ_∞(s) · L(s), where A^s for L_DH is (5/π)^{(s+1)/2}. Hmm so A^s isn't of the form A^s alone — there's a constant (5/π)^{1/2} multiplying. Let's write Λ_L(s) = c · (5/π)^{s/2} Γ((s+1)/2) L(s). Then A = √(5/π).
# Then 2 log A · (1/(2π))∫h(r) dr = log(5/π) · (1/(2π))∫h(r)dr.
# Plus the digamma contribution from Γ((s+1)/2) with κ=1: (1/(2π))∫ h(r) Re ψ((1+1+ir)/2/2)? Let's redo.
# 
# OK careful derivation. Take logarithmic derivative of Λ(s):
# Λ'/Λ(s) = log A + (γ_∞)'/γ_∞(s) + L'/L(s)
# For γ_∞(s) = Γ((s+κ)/2): log derivative = (1/2)ψ((s+κ)/2).
# 
# Explicit formula (for h holomorphic in horiz strip, even, decay):
# Σ_ρ h(γ_ρ) = pole_terms 
# + (1/(2π)) ∫_{-∞}^∞ h(r) [log A + (1/2)ψ(1/4+κ/2+ir/2) + ...]·(...) dr 
# 
# Actually the conventional symmetric form uses the FE to symmetrize:
# Σ_ρ h(γ_ρ) = (1/(2π)) ∫ h(r) [2 log A + Re ψ((1+κ+ir)/2)] dr - (1/π) Σ_n Λ_L(n)/√n ĥ(log n) + poles.
# (Convention: ĥ(u) = ∫ h(t) cos(tu) dt = 2 Re ∫_0^∞ h(t) e^{itu} dt for h even.)
# 
# So Ω_L(r) = 2 log A + Re ψ((1+κ+ir)/2).
# 
# For ζ: A = π^{-1/2}, κ=0. 2 log A = -log π. (1+0+ir)/2 = 1/2 + ir/2.
# Ω_ζ(r) = -log π + Re ψ(1/2 + ir/2).
# 
# WAIT — but we numerically validated Ω_ζ(r) = -log π + Re ψ(1/4 + ir/2). 
# So that means (1+κ+ir)/2 = 1/4+ir/2? Solving 1+κ = 1/2 ⟹ κ = -1/2. That doesn't match standard κ=0 for ζ.
# 
# So there's a discrepancy in my convention. Let me recheck.
# 
# Actually I think the formula uses ψ at 1/4 + ir/2, which corresponds to (s+κ)/2 evaluated at s=1/2+ir, i.e., (1/2+ir+κ)/2 = 1/4 + κ/2 + ir/2.
# For κ=0: 1/4 + ir/2. ✓
# For κ=1: 1/4 + 1/2 + ir/2 = 3/4 + ir/2. ✓
# 
# So Ω_L(r) = 2 log A + Re ψ(1/4 + κ/2 + ir/2). Not Re ψ((1+κ+ir)/2). My earlier formula was right.
# 
# So Ω_DH(r) = log(5/π) + Re ψ(3/4 + ir/2) = log 5 - log π + Re ψ(3/4 + ir/2). That's what I used. ✓
# 
# So where's the 4e-4 from? Let me check the off-line zero contribution more carefully — maybe more off-line zeros exist that I'm missing?
print("Re-checking off-line contribution.")
total_off = 0
for _, row in offline.iterrows():
 sig=row['sigma']; tt=row['t']
 z = complex(tt, -(sig-0.5))
 total_off += 4*np.real(np.exp(-z**2/(2*sigma_g**2)))
print("Total off-line contribution (110 zeros):", total_off)
# 3.4e-7. 


Σ critical (mpmath, just positive γ, then 2×): 17.262131380640490720529864574
First 10 L_DH zeros: [ 5.09415984 8.93991441 12.13354543 14.40400311 17.1302394 19.30880017
 22.15970777 23.34537011 26.09496735 27.92379882]
Re-checking off-line contribution.
Total off-line contribution (110 zeros): 3.359881051496222e-07


In [33]:
# Hmm. 4e-4 residual mismatch in L_DH explicit formula is puzzling given <1e-14 mismatch for ζ and L(χ_5).
# Possible causes:
# 1. We're missing some zeros (the 110 off-line is supposed to be definitive for t<5000, but maybe more exist beyond)
# 2. The DH zero list may include some inaccurate zeros (only 5e-15 |L_DH| at zero, given DOUBLE precision storage)
# 3. The "log 5 vs log(15) etc" - is the actual DH conductor 5?
# 
# Let me also try: maybe the DH conductor is 60 (5·12=?) — actually no, classical DH has FE with conductor 5.
# 
# Let me try without "log 5":
def arch_DH_no5(r):
 return h_gauss_z(r) * (sp_digamma(0.75 + 1j*r/2).real - np.log(np.pi))
val, _ = squad(arch_DH_no5, -200, 200, limit=500)
print("Arch DH (no log 5):", val/(2*np.pi))

# And with log 5:
def arch_DH_with5(r):
 return h_gauss_z(r) * (sp_digamma(0.75 + 1j*r/2).real + np.log(5) - np.log(np.pi))
val5, _ = squad(arch_DH_with5, -200, 200, limit=500)
print("Arch DH (with log 5):", val5/(2*np.pi))
print("LHS:", lhs_DH)
print("LHS - arch_no5:", lhs_DH - val/(2*np.pi))
print("LHS - arch_with5:", lhs_DH - val5/(2*np.pi))


Arch DH (no log 5): 4.421086373736694
Arch DH (with log 5): 17.262542992756323
LHS: 17.2621317166286
LHS - arch_no5: 12.841045342891904
LHS - arch_with5: -0.0004112761277248467


In [34]:
# Definitely needs log 5. The 4e-4 residual must be from accumulated truncation/precision in the LHS (zeros stored at 50 dp but only 14-digit double when cast to float).
# 5000 zeros, each at ~1e-15 random error in h(γ): 5000 errors of √5000·1e-15 ~ 7e-14. That's too small.
# 
# Could be many zeros have round-off accumulating: h(γ_n) for γ_n ≈ 14: ε in γ leads to ε·(γ/σ²)·h. With σ=20, γ=14: derivative ~ 14/400 · h ~ 0.035 h. h ≈ exp(-14²/800)=exp(-0.245)≈0.78. So derivative is 0.027. Float storage gives γ to ~1e-13 precision. So error per zero ~ 0.027·1e-13 = 3e-15. Sum 10000 zeros: ~3e-11. Still too small.
# 
# Actually, maybe σ=20 makes h's effective support extend to γ~80, which means we evaluate h at γ where 2.5σ relative error in float precision dominates. Recomputing with higher precision:
import mpmath as mp
mp.mp.dps = 30
ldh_g_mp30 = [mp.mpf(s) for s in ldh]
sigma_g = mp.mpf(20)
s30 = mp.mpf(0)
for g in ldh_g_mp30:
 s30 += mp.exp(-g**2/(2*sigma_g**2))
print("LHS critical at dps=30:", 2*s30)
# Compare to mp.mpf(20)**2 — sigma_g must be a high-precision number
# Already is.


LHS critical at dps=30: 17.262131380640490720529864574


In [35]:
# Same 17.262131. So zeros are stored correctly. Then the 4e-4 must reflect: 
# (a) MISSING zeros - e.g., are there 5000 zeros up to T=4981, or did we miss some?
# Riemann-von Mangoldt: N(T) ≈ (T/(2π)) log(T/(2π e)) + (1/π) log(q π / (2π e)^?). 
# For L_DH with conductor 5 (so analytic conductor C(t)=5/(π)... well effective):
# Expected number of zeros up to T=4981 for an L-function of degree 1, conductor 5: N(T) ≈ (T/(2π)) log(5 T/(2π e)).
# At T=4981: (4981/(2π))·log(5·4981/(2π·e)) = 793·log(1458) ≈ 793·7.28 ≈ 5773. 
# But we only have 5000 critical zeros (up to T=4981). So we're MISSING ~700 zeros! Many on critical line beyond what we have, or off-line.
# 
# Hmm. But descriptions said "first 5000 critical-line zeros to t≈5100." If there are ~5773 zeros up to T=4981 by Riemann-von Mangoldt, then we're missing ~700.
# 
# Wait Riemann-von Mangoldt for ζ: N(T) ≈ (T/(2π))log(T/(2πe))+7/8. For ζ at T=4981: ≈ 4981/(2π)·log(4981/(2πe))≈ 793·log(291)≈ 793·5.67 ≈ 4500. But we have 5000 ζ-zeros and they go up to γ_5000=5447. Consistent.
# 
# For L_DH degree 1 with conductor 5: density is higher by log(5)·1/(2π) per unit length. At T=4981: (T/(2π)) log(5T/(2πe)) ≈ 793·(log5 + log(291)) ≈ 793·(1.61 + 5.67) ≈ 793·7.28 ≈ 5773. 
# But L_DH zero list goes up to γ_5000 ≈ 4981. So consistent: we have 5000 zeros up to T=4981, but RvM says 5773 should exist by that height. We're missing ~770 zeros!
# 
# This is consistent with the 4e-4 residual: the 770 missing zeros at γ ~ 4000-5000, with h(γ)= exp(-γ²/800) ≈ exp(-20000) ~ 0. Wait that's negligible.
# 
# OK so RvM density mismatch: maybe DH zero list is INCOMPLETE near origin? Like maybe there are missing zeros at low γ that contribute substantially?
# 
# Anyway, our task is to test Weil positivity, NOT verify the explicit formula. Let's proceed and report this caveat.
# 
# Actually wait: my LHS computation might disagree with formula because the FORMULA itself sums over ALL zeros (countably infinite), and we only have a partial list. So mathematically LHS_truncated < RHS_full when h is concentrated near origin where zeros are dense. The 4e-4 then is exactly the missing-zero contribution from high-γ critical-line zeros, which exp(-γ²/800) for γ ~ 80 is exp(-8)≈3e-4. Aha, that's plausible!
# 
# σ=20: h decays as exp(-γ²/800). h(80)=exp(-8)≈3.4e-4. So zeros at γ ~ 80 contribute ~4e-4. We have ALL such zeros (γ_5000 ≈ 4981 >> 80), so they SHOULD be in our sum. Why don't they match?
# 
# OH WAIT — I see. The 4e-4 difference is small and roughly = h(85)·several = approx contribution from a missing batch of zeros NEAR γ ~ 80-100. Let me cross-check by counting DH zeros up to γ=100.
print("DH zeros up to γ=100:", np.sum(ldh_g < 100))
print("ζ zeros up to γ=100:", np.sum(zeta_g < 100))
# RvM for DH: N(100) ≈ (100/(2π))·log(5·100/(2π·e)) = 15.9·log(29.3) = 15.9·3.38 ≈ 53.8
# RvM for ζ: N(100) ≈ (100/(2π))·log(100/(2πe)) = 15.9·log(5.85) ≈ 15.9·1.77 ≈ 28.1


DH zeros up to γ=100: 52
ζ zeros up to γ=100: 29


In [36]:
# Counts match RvM closely (DH:52 vs predicted 54, ζ:29 vs predicted 28). So both lists complete up to γ=100. Good.
# 
# So why the 4e-4 residual for DH but not ζ?
# Maybe arch term has small error. Let me try MUCH wider integration ranges with scipy adaptive:
from scipy.integrate import quad
def arch_DH_fn(r):
 return np.exp(-r**2/(2*400)) * (sp_digamma(0.75 + 1j*r/2).real + np.log(5) - np.log(np.pi))
# Use trapezoidal with mpmath at high precision:
import mpmath as mp
mp.mp.dps = 60
def arch_DH_mp(r):
 rr = r
 return mp.exp(-rr**2/(2*mp.mpf(20)**2)) * (mp.re(mp.digamma(mp.mpf('0.75') + 1j*rr/2)) + mp.log(5) - mp.log(mp.pi))

# trapezoidal on fine grid from -300 to 300:
val_tr = mp.mpf(0)
N=10000
dr = mp.mpf(600)/N
for i in range(N+1):
 r = -300 + dr*i
 w = 0.5 if (i==0 or i==N) else 1
 val_tr += w*arch_DH_mp(r)
val_tr *= dr/(2*mp.pi)
print("Arch DH trapezoidal mpmath:", val_tr)


Arch DH trapezoidal mpmath: 17.2625429927563198252750006085555840662240441677758312251361


In [37]:
# Arch is 17.2625429927563198, LHS is 17.2621313806404907. Diff = 4.1212e-4.
# 
# So the discrepancy is REAL. Something's off in my formula or zero list completeness.
# 
# Let me try: maybe DH conductor is sqrt(5) (so q=5 enters as ½ log 5)? Or maybe DH conductor is (5/π)? Let me parametrize and find what conductor gives match.
# Replace log(5) with x. Then LHS - (x·(1/(2π))∫h(r)dr + arch_other) = 0 ⟹ x = (arch_other - LHS) / ((1/(2π))∫h(r)dr).
# (1/(2π))∫h(r)dr = (1/(2π))·σ√(2π) = σ/√(2π) = 20/√(2π) ≈ 7.9788
# 
# Compute arch_other = arch_DH minus log(5)·7.9788:
arch_other = val_tr - mp.log(5)*20/mp.sqrt(2*mp.pi)
print("arch_other:", arch_other)
# Discrepancy expressed as x = (LHS - arch_other)/(20/sqrt(2π))
LHS_mp = 2*s30
x = (LHS_mp - arch_other) / (20/mp.sqrt(2*mp.pi))
print("Required log of conductor:", x)
print("exp(x):", mp.exp(x))


arch_other: 4.42108637373669333810343376670218354377699289117054293833644
Required log of conductor: 1.60938632450571447838285162675859290460080067438653643574688
exp(x): 4.9997420670112419985115737848994448108859782897923623740872


In [38]:
# exp(x) = 4.99974 ≈ 5. So conductor IS 5. Difference of 0.000258 in conductor estimate from 5 corresponds to missing-zero error of order 4e-4 / 7.98 ~ 5e-5 / log 5 ~ small fraction. 
# In other words, our LHS truncation/precision missing ~5e-5 in effective log conductor accuracy = 2.6e-4/5 in conductor.
# 
# This means there must be a small amount of additional contribution missing from the zero sum, but we've established the formula is essentially correct (and the conductor is 5 as expected).
# 
# For our Weil positivity test, the IMPORTANT thing is to compute the matrix M = Arch + Pole - Prime (NOT using zeros at all), and check PSD.
# 
# Given the issue with LHS not quite matching, our 4e-4 mismatch for DH might also reflect the fact that some zeros are systematically missing. We will accept this as a data quality issue.
# 
# **Proceed with computing the arithmetic-side Weil matrix M_jk for each L-function.**

# Setup: X=2, J=5 basis functions g_j with ĝ_j(u) = (1-|u|/X)cos(jπu/X) on [-X,X].

import numpy as np
from scipy.special import digamma as sp_digamma
from scipy.integrate import quad as squad
from sympy import factorint, primerange

X = 2.0
J = 5

def ghat(u, j, X=X):
 if abs(u) > X: return 0.0
 return (1 - abs(u)/X) * np.cos(j*np.pi*u/X)

def g_jt(t, j, X=X):
 # g_j(t) for real t: g_j(t) = (1/π) ∫_0^X (1-u/X) cos(jπu/X) cos(tu) du
 return squad(lambda u: (1-u/X)*np.cos(j*np.pi*u/X)*np.cos(t*u), 0, X, limit=200)[0]/np.pi

def Gjk(x, j, k, X=X):
 # G_jk(x) = (1/(4π²)) (ĝ_j ⋆ ĝ_k)(x)
 if abs(x) > 2*X: return 0.0
 lo = max(-X, x-X); hi = min(X, x+X)
 if lo >= hi: return 0.0
 return squad(lambda u: ghat(u,j)*ghat(x-u,k), lo, hi, limit=200)[0]/(4*np.pi**2)

# Precompute g_j on a fine grid for arch integral
import functools

@functools.lru_cache(maxsize=None)
def gjt_cached(t, j):
 return g_jt(t, j)

# Arch matrix for a given L-function spec (Omega_L function):
def arch_matrix(Omega_L, R=80, npts=8000):
 # M^arch_jk = (1/(2π)) ∫_{-R}^R g_j(r) g_k(r) Ω_L(r) dr
 rs = np.linspace(-R, R, npts)
 dr = rs[1]-rs[0]
 # compute g_j(r) for each j on the grid (real-valued, even function so use absolute r)
 gj_vals = np.zeros((J, npts))
 for j in range(J):
 for i, r in enumerate(rs):
 gj_vals[j,i] = g_jt(r, j)
 omega_vals = np.array([Omega_L(r) for r in rs])
 M = np.zeros((J,J))
 for j in range(J):
 for k in range(j,J):
 integrand = gj_vals[j]*gj_vals[k]*omega_vals
 v = np.trapz(integrand, rs)/(2*np.pi)
 M[j,k]=v; M[k,j]=v
 return M

print("Setup complete.")


Setup complete.


In [39]:
# Note about precomputing g_j on grid: This will be slow because squad is called npts*J times. Let me use sampling and vectorize.
# Actually g_j(r) = (1/π) ∫_0^X (1-u/X) cos(jπu/X) cos(ru) du. We can compute this analytically.
# 
# Let f_j(u) = (1-u/X) cos(jπu/X) on [0,X]. Then g_j(r) = (1/π) ∫_0^X f_j(u) cos(ru) du.
# Use cos(ru) = (e^{iru}+e^{-iru})/2. Then g_j(r) = (1/π) Re ∫_0^X f_j(u) e^{iru} du.
# For each j, f_j is a polynomial-trig combination — closed form exists.
# 
# Let me just precompute on a dense grid using scipy's quad for speed reasons:

import numpy as np
from scipy.integrate import quad
import time

X = 2.0

# Speed up: use a polynomial closed form
# ∫_0^X (1-u/X) cos(au) cos(bu) du where a=jπ/X, b=r.
# Product-to-sum: cos(au)cos(bu) = (cos((a-b)u)+cos((a+b)u))/2
# ∫(1-u/X) cos(cu) du = sin(cu)/c - (1/X)·[u sin(cu)/c + cos(cu)/c²]
# Let I(c) = sin(cu)/c - (u sin(cu) + cos(cu)/c)/(X·c) evaluated 0 to X.
# At u=0: sin=0, cos=1, so I(c)|_0 = -(1/(X·c²))·1 = -1/(X·c²).
# At u=X: sin(cX)/c - (X sin(cX) + cos(cX)/c)/(X·c) = sin(cX)/c - sin(cX)/c - cos(cX)/(X c²) = -cos(cX)/(X c²).
# So ∫_0^X (1-u/X) cos(cu) du = [-cos(cX)/(X c²)] - [-1/(X c²)] = (1 - cos(cX))/(X c²).
# (For c=0: limit is X/2.)
# 
# So:
# g_j(r) = (1/(2π))[∫(1-u/X) cos((a-r)u) du + ∫(1-u/X) cos((a+r)u) du]
# = (1/(2π))[(1-cos((a-r)X))/(X(a-r)²) + (1-cos((a+r)X))/(X(a+r)²)]
# where a = jπ/X.

def g_j_closed(r, j, X=X):
 a = j*np.pi/X
 out = 0.0
 for sign in [1,-1]:
 c = a + sign*r
 if abs(c) < 1e-12:
 out += X/2 # limit
 else:
 out += (1 - np.cos(c*X))/(X*c**2)
 return out/(2*np.pi)

# Cross-check with numerical integration
for j in [0,1,2,3]:
 for r in [0.0, 1.0, 3.0, 10.0]:
 a = g_j_closed(r,j); b = g_jt(r,j) if False else g_jt(r,j)
 # Use the slow numerical for comparison
 b = quad(lambda u: (1-u/X)*np.cos(j*np.pi*u/X)*np.cos(r*u),0,X)[0]/np.pi
 print(f"j={j} r={r}: closed={a:.8f} numerical={b:.8f} diff={a-b:.2e}")


j=0 r=0.0: closed=0.31830989 numerical=0.31830989 diff=0.00e+00
j=0 r=1.0: closed=0.22538677 numerical=0.22538677 diff=0.00e+00
j=0 r=3.0: closed=0.00070434 numerical=0.00070434 diff=1.03e-17
j=0 r=10.0: closed=0.00094207 numerical=0.00094207 diff=-6.18e-18
j=1 r=0.0: closed=0.12900614 numerical=0.12900614 diff=0.00e+00
j=1 r=1.0: closed=0.14963396 numerical=0.14963396 diff=-2.78e-17
j=1 r=3.0: closed=0.08383144 numerical=0.08383144 diff=-1.39e-17
j=1 r=10.0: closed=0.00241398 numerical=0.00241398 diff=1.52e-17
j=2 r=0.0: closed=0.00000000 numerical=-0.00000000 diff=4.42e-18
j=2 r=1.0: closed=0.03114108 numerical=0.03114108 diff=-1.04e-17
j=2 r=3.0: closed=0.15817821 numerical=0.15817821 diff=0.00e+00
j=2 r=10.0: closed=0.00127414 numerical=0.00127414 diff=-9.32e-18
j=3 r=0.0: closed=0.01433402 numerical=0.01433402 diff=-1.04e-17
j=3 r=1.0: closed=0.00479505 numerical=0.00479505 diff=7.81e-18
j=3 r=3.0: closed=0.05581846 numerical=0.05581846 diff=6.94e-18
j=3 r=10.0: closed=0.00452540 

In [40]:
# Closed form works. Also for complex r (needed for off-line zeros in zero-side sanity check):
def g_j_closed_z(z, j, X=X):
 """g_j evaluated at possibly-complex z."""
 a = j*np.pi/X
 out = 0.0+0.0j
 for sign in [1,-1]:
 c = a + sign*z
 if abs(c) < 1e-12:
 out += X/2
 else:
 out += (1 - np.cos(c*X))/(X*c**2)
 return out/(2*np.pi)

# Test
print(g_j_closed_z(1+0.5j, 1))
# Compare with our complex integration
print(g_j(1+0.5j, 1)) # from earlier function (slow)


(0.15211083434454867+0.016531576538753555j)
(0.15211083434454867-0.01653157653875356j)


In [41]:
# Sign differs! cos(complex)... Let me check sign convention.
# Our earlier g_j(t) for complex t used: cos(au) cosh(bu) - i sin(au) sinh(bu) for cos((a+ib)u), where t=a+ib.
# So the imaginary part of g_j(t) for t=a+ib should be: -(1/π)∫(1-u/X)cos(jπu/X)·sin(au)sinh(bu) du.
# 
# In closed form: cos(c·X) where c=a+r (with z=r=a+ib): c·X = (a + (real)+i(imag))·X. Hmm I used z directly. For z=a+ib, the formula (1-cos(cX))/(Xc²) should give complex value.
# 
# Compute (1-cos((jπ/X+z)X))/(X·(jπ/X+z)²):
# Let's recheck for j=1, z=1+0.5j:
import cmath
j=1
a=j*np.pi/X
z=1+0.5j
for sign in [1,-1]:
 c = a + sign*z
 val = (1 - cmath.cos(c*X))/(X*c**2)
 print(f"sign={sign}, c={c}, val={val}")
out = sum([(1-cmath.cos((a+sign*z)*X))/(X*(a+sign*z)**2) for sign in [1,-1]])/(2*np.pi)
print("Sum:", out)


sign=1, c=(2.5707963267948966+0.5j), val=(-0.005012075901490601-0.08199704301604895j)
sign=-1, c=(0.5707963267948966-0.5j), val=(0.9607526353179867+0.18586800182886004j)
Sum: (0.15211083434454867+0.016531576538753555j)


In [42]:
# Note np.cos for complex z does NOT give same as cmath.cos? Let me check.
print("np.cos(2+1j):", np.cos(2+1j))
print("cmath.cos(2+1j):", cmath.cos(2+1j))
# np uses cmath for complex, so they should agree.
# In my function I used np.cos. Let me verify.
def g_j_closed_z_v2(z, j, X=X):
 a = j*np.pi/X
 out = 0.0+0.0j
 for sign in [1,-1]:
 c = a + sign*z
 if abs(c) < 1e-12:
 out += X/2
 else:
 out += (1 - cmath.cos(c*X))/(X*c**2)
 return out/(2*np.pi)
print("v2:", g_j_closed_z_v2(1+0.5j, 1))
# vs np version
print("np ver:", g_j_closed_z(1+0.5j, 1))
# Likely numerically same when np.cos called with complex arg


np.cos(2+1j): (-0.64214812471552-1.0686074213827783j)
cmath.cos(2+1j): (-0.64214812471552-1.0686074213827783j)
v2: (0.15211083434454867+0.016531576538753555j)
np ver: (0.15211083434454867+0.016531576538753555j)


In [43]:
# So closed form gives (0.152, +0.0165) but the numerical integration via my earlier function gave (0.152, -0.0165).
# The difference is in the sign of imaginary part.
# 
# Question: which is the correct "g_j(t)" for complex t such that g_j extends analytically from the real axis?
# g_j(t) := (1/(2π)) ∫_{-X}^X ĝ_j(u) e^{itu} du. For real t this is real (since ĝ_j even, integral becomes (1/π)∫ĝ_j(u) cos(tu) du).
# For COMPLEX t, the analytic continuation has e^{itu} = e^{i(a+ib)u} = e^{iau}·e^{-bu}.
# 
# So: g_j(a+ib) = (1/(2π)) ∫_{-X}^X ĝ_j(u) e^{i(a+ib)u} du = (1/(2π)) ∫_{-X}^X ĝ_j(u) e^{-bu} (cos(au)+i sin(au)) du.
# ĝ_j even, so the imaginary part: (1/(2π)) ∫ĝ_j(u) e^{-bu} sin(au) du = (over symmetric u): 
# sin(au) is odd; e^{-bu} contributes asymmetrically; ĝ_j(u) even. Product = ĝ_j(u)·e^{-bu}·sin(au), odd if e^{-bu} were symmetric, but it's not.
# Actually e^{-bu} ĝ_j(u) sin(au) — replacing u→-u gives -e^{bu}ĝ_j(u)sin(au). So integral ≠ 0.
# 
# So the analytic continuation has REAL imaginary part depending on b.
# 
# My closed form: I derived (1-cos(cX))/(Xc²) for ∫_0^X (1-u/X)cos(cu) du, which assumes c is REAL.
# For complex c, this needs care — the derivation used ∫cos(cu)du = sin(cu)/c, and that's valid for complex c (entire function).
# So the closed form is correct for c complex (z is c−a part).
# 
# Where then is the sign discrepancy with my earlier complex-arg numerical integration?
# Earlier function:
# g_j(t complex) = (integ_re + 1j*integ_im)/π 
# with integ_re = ∫(1-u/X)cos(jπu/X)·cos(t.re·u)·cosh(t.im·u) du
# integ_im = ∫(1-u/X)cos(jπu/X)·sin(t.re·u)·sinh(t.im·u) du ← note no minus sign

# But cos(t·u) for complex t = cos((a+ib)u) = cos(au)cosh(bu) - i sin(au) sinh(bu).
# So cos(t·u) has imaginary part -sin(au)sinh(bu). When integrated this gives:
# ∫cos(t·u)·... du has imaginary part = -∫sin(au)sinh(bu)·... du.
# 
# So g_j(t)_complex = (1/π) ∫ (1-u/X)cos(jπu/X) cos(t·u) du
# has Im part = -(1/π) ∫(1-u/X)cos(jπu/X) sin(t.re·u) sinh(t.im·u) du.
# 
# Earlier I had integ_im = ∫(1-u/X)cos(jπu/X)·sin(t.re·u)·sinh(t.im·u) du (no minus sign), giving g_j = (integ_re + 1j*integ_im)/π. The MISSING minus sign is the bug.
# 
# The closed form is correct. The complex numerical version had a sign bug.
# 
# Let me fix and trust the closed form:
print("Closed form is correct.")

# But wait: the explicit formula sums φ(γ_ρ) where γ_ρ = (ρ - 1/2)/i. For off-line ρ = β+iγ:
# γ_ρ = (β+iγ - 1/2)/i = (β - 1/2)/i + γ = γ - i(β - 1/2).
# So we evaluate g_j at z = γ - i(β-1/2). For β>1/2, b = -(β-1/2) < 0.
# 
# In our test (off-line zeros σ>1/2): z = t - i(σ-1/2), so b = -(σ-1/2) < 0.
# 
# OK now we have the right machinery. Move on.


Closed form is correct.


In [44]:
# Now build the Weil matrix M for each L-function.
# M_jk = Arch_jk(L) + Pole_jk(L) - Prime_jk(L)
# 
# Setup: J=5 basis functions with X=2. Then primes/n contributing have log n ≤ 2X = 4 ⟹ n ≤ 54.

import numpy as np
from scipy.special import digamma as sp_digamma
import scipy.integrate as sint
from sympy import factorint

X = 2.0
J = 5

def g_j_closed(r, j, X=X):
 a = j*np.pi/X
 out = 0.0
 for sign in [1,-1]:
 c = a + sign*r
 if abs(c) < 1e-12: out += X/2
 else: out += (1 - np.cos(c*X))/(X*c**2)
 return out/(2*np.pi)

def g_j_closed_c(z, j, X=X):
 a = j*np.pi/X
 out = 0.0+0.0j
 for sign in [1,-1]:
 c = a + sign*z
 if abs(c) < 1e-12: out += X/2
 else: out += (1 - np.cos(c*X))/(X*c**2)
 return out/(2*np.pi)

def Gjk_conv(x, j, k, X=X):
 """G_jk(x) = (1/(4π²)) (ĝ_j ⋆ ĝ_k)(x)"""
 if abs(x) > 2*X: return 0.0
 lo = max(-X, x-X); hi = min(X, x+X)
 if lo >= hi: return 0.0
 def integrand(u):
 return (1 - abs(u)/X)*np.cos(j*np.pi*u/X) * (1 - abs(x-u)/X)*np.cos(k*np.pi*(x-u)/X)
 return sint.quad(integrand, lo, hi, limit=200)[0]/(4*np.pi**2)

# Pre-compute g_j on grid for arch integral
R_int = 80.0 # truncation for arch integral
npts = 2401
r_grid = np.linspace(-R_int, R_int, npts)
gj_grid = np.zeros((J, npts))
for j in range(J):
 for i, r in enumerate(r_grid):
 gj_grid[j, i] = g_j_closed(r, j)

# Helper: arch matrix given Omega_L function
def compute_arch_matrix(Omega_L):
 omega_vals = np.array([Omega_L(r) for r in r_grid])
 M = np.zeros((J,J))
 for j in range(J):
 for k in range(j,J):
 integrand = gj_grid[j] * gj_grid[k] * omega_vals
 v = np.trapz(integrand, r_grid)/(2*np.pi)
 M[j,k] = v; M[k,j] = v
 return M

# Prime sum matrix given Λ_L(n) array (indexed by n) and any other arithmetic data:
def compute_prime_matrix(Lambda_L, n_range):
 # Prime_jk = 2 Σ_n Λ_L(n)/√n · G_jk(log n)
 M = np.zeros((J,J))
 for j in range(J):
 for k in range(j,J):
 s = 0.0
 for n in n_range:
 if Lambda_L[n] != 0:
 s += Lambda_L[n]/np.sqrt(n) * Gjk_conv(np.log(n), j, k)
 M[j,k] = 2*s; M[k,j] = M[j,k]
 return M

# Pole matrix
def compute_pole_matrix(z_pole=None, factor=1.0):
 # For ζ: pole at s=1 gives contribution h(i/2) + h(-i/2) = 2 h(i/2) [h even], i.e. 2 g_j(i/2) g_k(i/2)
 M = np.zeros((J,J))
 if z_pole is None: return M
 z = z_pole
 g_vec = np.array([g_j_closed_c(z, j) for j in range(J)])
 # Since z=i/2, g_j(z) is real (for our even g_j, g_j(ib) is real because cos(c(ib))=cosh(...))
 # Actually g_j(i/2) is real because g_j is even Schwartz and analytic, and g_j(it) is real for real t (Fourier transform of real even function).
 g_real = np.array([gj.real for gj in g_vec])
 M = factor * np.outer(g_real, g_real)
 return M

print("Helpers defined.")


Helpers defined.


In [45]:
# Verify Gjk via FT consistency
print("G_00(0):", Gjk_conv(0,0,0)) # ~0.0338
print("G_00(0) check via FT:", sint.quad(lambda r: g_j_closed(r,0)**2, -50, 50)[0]/(2*np.pi))
print("G_11(0):", Gjk_conv(0,1,1), "vs FT:", sint.quad(lambda r: g_j_closed(r,1)**2, -50, 50)[0]/(2*np.pi))


G_00(0): 0.033773727880779265
G_00(0) check via FT: 0.03377369618322611
G_11(0): 0.019453359504060713 vs FT: 0.01945332629393744


In [46]:
# Now compute matrices for each L-function.
# 
# 1) ζ: Ω(r) = -log π + Re ψ(1/4 + ir/2). Λ_ζ(n) = Λ(n) usual von Mangoldt. Pole at z=i/2.
# 2) L(χ_5) [real quadratic χ_5, EVEN char]: Ω(r) = log 5 - log π + Re ψ(1/4+ir/2). Λ_χ(n) = Λ(n)·χ_5(n).
# 3) L_DH: Ω(r) = log 5 - log π + Re ψ(3/4+ir/2). Λ_DH(n) from Dirichlet inversion (no Euler product). No pole.
# 
# Note: L(Δ,s) Ramanujan and ζ_δ data unavailable; will discuss in limitations.
# 
# For matrices we DON'T USE zeros — purely arithmetic side. So no need for zero lists for matrix construction.
# We use the zero lists only as a consistency check.

# 1) ζ
def Omega_zeta(r): return -np.log(np.pi) + sp_digamma(0.25 + 1j*r/2).real

# Λ for ζ
Lambda_z = np.zeros(60)
for n in range(2, 60):
 fac = factorint(n)
 if len(fac)==1:
 p, _ = next(iter(fac.items()))
 Lambda_z[n] = np.log(p)
n_range = range(2, 60)

# 2) L(χ_5)
chi5 = {0:0, 1:1, 2:-1, 3:-1, 4:1}
def chi5val(n): return chi5[n%5]
def Omega_chi5(r): return np.log(5) - np.log(np.pi) + sp_digamma(0.25 + 1j*r/2).real
Lambda_chi5 = np.zeros(60)
for n in range(2, 60):
 fac = factorint(n)
 if len(fac)==1:
 p, k = next(iter(fac.items()))
 cv = chi5val(p)**k
 Lambda_chi5[n] = np.log(p)*cv

# 3) L_DH
alpha_f = float((np.sqrt(10-2*np.sqrt(5))-2)/(np.sqrt(5)-1))
a_DH_arr = np.zeros(60)
for n in range(1, 60):
 r = n%5
 a_DH_arr[n] = {0:0.0,1:1.0,2:alpha_f,3:-alpha_f,4:-1.0}[r]
Lambda_DH_arr = np.zeros(60)
for n in range(2,60):
 val = a_DH_arr[n]*np.log(n)
 for d in range(2,n+1):
 if n%d==0 and d<n:
 val -= a_DH_arr[d]*Lambda_DH_arr[n//d]
 Lambda_DH_arr[n] = val

def Omega_DH(r): return np.log(5) - np.log(np.pi) + sp_digamma(0.75 + 1j*r/2).real

# Compute matrices
print("Computing ζ matrices...")
M_arch_z = compute_arch_matrix(Omega_zeta)
M_prime_z = compute_prime_matrix(Lambda_z, n_range)
M_pole_z = compute_pole_matrix(z_pole=0.5j, factor=2.0) # 2 h(i/2) = 2 g_j(i/2) g_k(i/2)
M_z = M_arch_z + M_pole_z - M_prime_z

print("Computing L(χ_5) matrices...")
M_arch_chi = compute_arch_matrix(Omega_chi5)
M_prime_chi = compute_prime_matrix(Lambda_chi5, n_range)
M_chi = M_arch_chi - M_prime_chi # no pole

print("Computing L_DH matrices...")
M_arch_DH = compute_arch_matrix(Omega_DH)
M_prime_DH = compute_prime_matrix(Lambda_DH_arr, n_range)
M_DH = M_arch_DH - M_prime_DH # no pole

print("Eigenvalues:")
print("ζ:", np.linalg.eigvalsh(M_z))
print("L(χ_5):", np.linalg.eigvalsh(M_chi))
print("L_DH:", np.linalg.eigvalsh(M_DH))


Computing ζ matrices...


AttributeError: module 'numpy' has no attribute 'trapz'

In [47]:
import numpy as np
# Use np.trapezoid (newer NumPy):
def compute_arch_matrix(Omega_L):
 omega_vals = np.array([Omega_L(r) for r in r_grid])
 M = np.zeros((J,J))
 for j in range(J):
 for k in range(j,J):
 integrand = gj_grid[j] * gj_grid[k] * omega_vals
 v = np.trapezoid(integrand, r_grid)/(2*np.pi)
 M[j,k] = v; M[k,j] = v
 return M

print("Computing ζ matrices...")
M_arch_z = compute_arch_matrix(Omega_zeta)
M_prime_z = compute_prime_matrix(Lambda_z, n_range)
M_pole_z = compute_pole_matrix(z_pole=0.5j, factor=2.0)
M_z = M_arch_z + M_pole_z - M_prime_z

print("Computing L(χ_5) matrices...")
M_arch_chi = compute_arch_matrix(Omega_chi5)
M_prime_chi = compute_prime_matrix(Lambda_chi5, n_range)
M_chi = M_arch_chi - M_prime_chi

print("Computing L_DH matrices...")
M_arch_DH = compute_arch_matrix(Omega_DH)
M_prime_DH = compute_prime_matrix(Lambda_DH_arr, n_range)
M_DH = M_arch_DH - M_prime_DH

print("Eigenvalues of M (Weil quadratic form matrix):")
print("ζ: ", np.linalg.eigvalsh(M_z))
print("L(χ_5):", np.linalg.eigvalsh(M_chi))
print("L_DH: ", np.linalg.eigvalsh(M_DH))


Computing ζ matrices...
Computing L(χ_5) matrices...
Computing L_DH matrices...


Eigenvalues of M (Weil quadratic form matrix):
ζ: [-1.37035193e-10 7.96556791e-10 7.94385445e-08 2.02243042e-06
 3.24014324e-05]
L(χ_5): [5.65115441e-09 4.65067561e-07 7.93237844e-06 9.25243002e-05
 4.94741214e-02]
L_DH: [1.27586457e-08 1.16642964e-06 2.61046714e-05 3.31933515e-05
 6.86201424e-02]


In [48]:
# Interesting!
# - For ζ: λ_min ≈ -1.37e-10 (essentially zero within numerical precision, could be just integration error)
# - For χ_5: λ_min ≈ +5.65e-9 (positive, very small)
# - For L_DH: λ_min ≈ +1.28e-8 (positive!)
# 
# All eigenvalues are tiny (~10^-10 to 10^-8 for smallest) — this suggests the matrices are nearly rank-deficient (which makes sense: with finite truncation X=2, the "small" eigenvalues correspond to test functions whose Fourier support primarily lies near the boundary of [-X,X] and have small g-values).
# 
# CRUCIALLY: L_DH does NOT show negative λ_min. All three matrices are essentially PSD numerically.
# 
# Why? Recall the Weil matrix equals (by explicit formula identity) Σ_ρ g(γ_ρ) g_k(γ_ρ) over ALL zeros. For:
# - GRH-satisfying L: all γ real, sum is PSD ✓
# - L_DH with off-line zeros: complex γ_ρ contributes. But the off-line zeros have height t ≥ 114 (per CSV), and our g_j decays rapidly (compactly supported FT, so g_j(r) ~ 1/r²). At r=114, g_0(114) ≈ (1/(2π))·(1-cos(228))/(2·114²) ≈ 2/(2π·26000) ≈ 1.2e-5. Squared: 1.4e-10. With 110 off-line zeros: 1.5e-8. Comparable to λ_min!
# 
# So the off-line contribution IS at the level of λ_min, but doesn't push it negative because:
# (a) g_j(γ-i(σ-1/2)) for small σ-1/2 is ≈ g_j(γ) (analytic continuation, small shift), still mostly positive contribution.
# (b) With our basis (5 functions, X=2), we don't have enough resolution / specifically tuned test function to "concentrate" near off-line zeros to make the form negative.

# To make this rigorous let me extend the basis (J larger) and try larger X to enable resolving complex shift better:
print("Try J=10, X=4")
X = 4.0
J = 10
n_range = range(2, int(np.exp(2*X))+1) # primes/n with log n ≤ 8 ⟹ n ≤ 2981

# Re-precompute g_j on grid
R_int = 80.0
npts = 4001
r_grid = np.linspace(-R_int, R_int, npts)
gj_grid = np.zeros((J, npts))
for j in range(J):
 for i, r in enumerate(r_grid):
 gj_grid[j, i] = g_j_closed(r, j)

# Rebuild Λ arrays for n_range up to 2981
N_TOP = int(np.exp(2*X))+1
print("N_TOP:", N_TOP)

# ζ
Lambda_z = np.zeros(N_TOP+1)
for n in range(2, N_TOP+1):
 fac = factorint(n)
 if len(fac)==1:
 p, _ = next(iter(fac.items()))
 Lambda_z[n] = np.log(p)
# χ_5
Lambda_chi5 = np.zeros(N_TOP+1)
for n in range(2, N_TOP+1):
 fac = factorint(n)
 if len(fac)==1:
 p, k = next(iter(fac.items()))
 cv = chi5val(p)**k
 Lambda_chi5[n] = np.log(p)*cv
# DH
a_DH_arr2 = np.zeros(N_TOP+1)
for n in range(1, N_TOP+1):
 r = n%5
 a_DH_arr2[n] = {0:0.0,1:1.0,2:alpha_f,3:-alpha_f,4:-1.0}[r]
Lambda_DH_arr2 = np.zeros(N_TOP+1)
for n in range(2, N_TOP+1):
 val = a_DH_arr2[n]*np.log(n)
 for d in range(2,n+1):
 if n%d==0 and d<n:
 val -= a_DH_arr2[d]*Lambda_DH_arr2[n//d]
 Lambda_DH_arr2[n] = val

print("Arithmetic data ready.")


Try J=10, X=4
N_TOP: 2981


Arithmetic data ready.


In [49]:
# Compute matrices with new (X=4, J=10) basis. Be efficient about prime sum.
import time

n_range = range(2, N_TOP+1)

t0 = time.time()
print("Computing arch matrices...")
M_arch_z = compute_arch_matrix(Omega_zeta)
M_arch_chi = compute_arch_matrix(Omega_chi5)
M_arch_DH = compute_arch_matrix(Omega_DH)
print(f"Arch done ({time.time()-t0:.1f}s)")

# Optimize prime sum: precompute G_jk(log n) for all (j,k,n) once
print("Computing prime matrices...")
# Cache G values
t0 = time.time()
G_table = {} # G_table[(j,k,n)] = G_jk(log n)
# Compute via vectorized direct integration on a fine u-grid (much faster than scipy.quad per (j,k,n))
n_arr = np.arange(2, N_TOP+1)
log_n_arr = np.log(n_arr)
sqrt_n_arr = np.sqrt(n_arr)

# Precompute u-grid for convolution
u_grid = np.linspace(-X, X, 1601)
du = u_grid[1]-u_grid[0]
ghat_grid = np.zeros((J, len(u_grid)))
for j in range(J):
 ghat_grid[j] = (1 - np.abs(u_grid)/X) * np.cos(j*np.pi*u_grid/X) * (np.abs(u_grid)<=X)

# G_jk(x) = (1/(4π²)) ∫_{-X}^X ĝ_j(u) ĝ_k(x-u) du
# For each n (and thus x = log n), evaluate via interpolation/grid integration
# More efficient: define G_jk(x) numerically via convolution-grid evaluation
import scipy.interpolate as si

# Precompute interpolants for ĝ_k at arbitrary point
def ghat_eval(u_arr, k):
 return (np.abs(u_arr)<=X) * (1 - np.abs(np.clip(u_arr,-X,X))/X) * np.cos(k*np.pi*u_arr/X)

def Gjk_fast(x, j, k):
 if abs(x) >= 2*X: return 0.0
 # Integrate over u in [-X, X] ∩ [x-X, x+X]:
 lo = max(-X, x-X); hi = min(X, x+X)
 nint = 401
 us = np.linspace(lo, hi, nint)
 v1 = (1 - np.abs(us)/X) * np.cos(j*np.pi*us/X)
 v2 = ghat_eval(x-us, k)
 return np.trapezoid(v1*v2, us)/(4*np.pi**2)

# Compute G_jk(log n) for all (j,k,n)
G_arr = np.zeros((J, J, len(n_arr)))
for j in range(J):
 for k in range(j, J):
 for i, n in enumerate(n_arr):
 ln = log_n_arr[i]
 if ln >= 2*X: continue
 G_arr[j,k,i] = Gjk_fast(ln, j, k)
 G_arr[k,j] = G_arr[j,k]
print(f"G table done ({time.time()-t0:.1f}s)")

def prime_matrix(Lambda_arr):
 M = np.zeros((J,J))
 weights = Lambda_arr[2:N_TOP+1] / sqrt_n_arr
 for j in range(J):
 for k in range(j,J):
 M[j,k] = 2*np.sum(weights * G_arr[j,k])
 M[k,j] = M[j,k]
 return M

t0 = time.time()
M_prime_z = prime_matrix(Lambda_z)
M_prime_chi = prime_matrix(Lambda_chi5)
M_prime_DH = prime_matrix(Lambda_DH_arr2)
print(f"Prime done ({time.time()-t0:.1f}s)")

M_pole_z = compute_pole_matrix(z_pole=0.5j, factor=2.0)
M_z = M_arch_z + M_pole_z - M_prime_z
M_chi = M_arch_chi - M_prime_chi
M_DH = M_arch_DH - M_prime_DH

print()
print("Eigenvalues of M (X=4, J=10):")
print("ζ: ", np.linalg.eigvalsh(M_z))
print("L(χ_5):", np.linalg.eigvalsh(M_chi))
print("L_DH: ", np.linalg.eigvalsh(M_DH))


Computing arch matrices...
Arch done (0.0s)
Computing prime matrices...


G table done (8.7s)
Prime done (0.0s)

Eigenvalues of M (X=4, J=10):
ζ: [-1.17540987e+00 1.89396620e-04 1.83610514e-03 4.72193419e-03
 9.10201735e-03 1.47111694e-02 2.19509793e-02 3.16382938e-02
 4.78667408e-02 9.42765704e-02]
L(χ_5): [-2.10790940e-02 -1.02730623e-02 -5.36799398e-03 -2.55930597e-03
 -5.83031928e-04 -5.31449310e-06 2.56753746e-03 1.61147827e-02
 1.57929223e-01 2.41657589e-01]
L_DH: [-2.61858297e-02 -1.03778655e-02 -7.11964338e-03 -2.32050976e-03
 -5.24335333e-04 -2.70407915e-05 1.63780149e-03 9.34340916e-03
 3.67250801e-02 2.50253749e-01]


In [50]:
# Now ALL three functions have NEGATIVE minimum eigenvalues. This is a sign of truncation/approximation error in the matrix, 
# not a sign of GRH failure.
# 
# Crucial diagnostic: for ζ, λ_min = -1.175 — this should be 0 under GRH! So our matrix computation has a significant systematic error.
# 
# Most likely sources:
# 1. Truncation of arch integral (R_int=80)
# 2. Numerical integration error in G_jk
# 3. Truncation of prime sum at n ≤ 2981 (log n ≤ 8 = 2X). This SHOULD capture everything since G_jk(log n)=0 for log n ≥ 2X. But Gjk_fast uses 401 points — interpolation could be inaccurate.
# 4. The pole contribution: 2 g_j(i/2) g_k(i/2). I computed g_j(i/2) using closed form. For pole at s=1, we want h(γ_ρ) at ρ=1 i.e. γ_ρ = (1-1/2)/i = -i/2. So evaluate h at -i/2, which (since h even) equals h(i/2). Plus pole at s=0 contributes h evaluated at (-1/2)/i = i/2. Total: 2 h(i/2). ✓.
# 
# Let me first check that the explicit-formula identity holds in our matrix construction: M_jk should equal Σ_ρ g_j(γ_ρ) g_k(γ_ρ) over ALL zeros, where for L_DH only γ values include complex shifts.
# 
# Using our truncated zero lists:
def zero_sum_matrix(g_real_at_zeros):
 M = np.zeros((J, J))
 for j in range(J):
 for k in range(j, J):
 M[j,k] = 2*np.sum(g_real_at_zeros[j]*g_real_at_zeros[k])
 M[k,j] = M[j,k]
 return M

# zeta zeros
print("Building zero-sum matrix for ζ...")
gj_at_zeta = np.zeros((J, len(zeta_g)))
for j in range(J):
 for i, gam in enumerate(zeta_g):
 gj_at_zeta[j, i] = g_j_closed(gam, j)
M_zero_zeta = zero_sum_matrix(gj_at_zeta)
eig_zero_zeta = np.linalg.eigvalsh(M_zero_zeta)
print("Zero-sum eigenvalues ζ:", eig_zero_zeta)
print("vs arithmetic M_z eigenvalues ζ:", np.linalg.eigvalsh(M_z))
print()
print("M_zero_zeta - M_z eigenvalues:", np.linalg.eigvalsh(M_zero_zeta - M_z))


Building zero-sum matrix for ζ...
Zero-sum eigenvalues ζ: [1.30380631e-19 2.68095755e-16 6.14567340e-15 8.22207778e-12
 1.11542494e-10 3.69697442e-08 1.20070110e-07 5.52494338e-06
 9.14343727e-06 5.92424921e-02]
vs arithmetic M_z eigenvalues ζ: [-1.17540987e+00 1.89396620e-04 1.83610514e-03 4.72193419e-03
 9.10201735e-03 1.47111694e-02 2.19509793e-02 3.16382938e-02
 4.78667408e-02 9.42765704e-02]

M_zero_zeta - M_z eigenvalues: [-9.42486842e-02 -4.76687120e-02 -3.08388611e-02 -1.99181001e-02
 -1.17667454e-02 -5.74832711e-03 -2.00455118e-03 -1.84874679e-04
 4.53330387e-02 1.17541980e+00]


In [51]:
# The zero-sum matrix for ζ has all-PSD eigenvalues (min = 1.3e-19), as expected (it's a Gram matrix).
# But it does NOT match M_z (arithmetic side). Large discrepancy (eigenvalues differ by ~1.18).
# 
# This means our matrix M_jk = Arch + Pole - Prime has a SIGNIFICANT systematic error.
# 
# Let me check by computing for a SINGLE chosen test function whether explicit formula holds:
# Take φ = g_0 · g_0 (i.e., j=k=0). Then 
# LHS = 2 Σ_{γ>0} g_0(γ)² (zero sum)
# RHS = M_arch[0,0] + M_pole[0,0] - M_prime[0,0]

# Already computed:
print("LHS (zero sum) j=k=0:", M_zero_zeta[0,0])
print("Arch[0,0]:", M_arch_z[0,0])
print("Pole[0,0]:", M_pole_z[0,0])
print("Prime[0,0]:", M_prime_z[0,0])
print("RHS (arith) = arch+pole-prime:", M_arch_z[0,0] + M_pole_z[0,0] - M_prime_z[0,0])
print("Diff:", M_zero_zeta[0,0] - (M_arch_z[0,0] + M_pole_z[0,0] - M_prime_z[0,0]))


LHS (zero sum) j=k=0: 5.919477305807289e-06
Arch[0,0]: -0.11020225835038579
Pole[0,0]: 0.23906658376603174
Prime[0,0]: 1.262206966343379
RHS (arith) = arch+pole-prime: -1.133342640927733
Diff: 1.1333485604050388


In [52]:
# Huge mismatch: prime sum is 1.26 but should be small. Let me check by direct computation what 2 Σ Λ(n)/√n · G_00(log n) is.
#
# G_00(log 2) = ?
print("G_00(log 2):", Gjk_fast(np.log(2), 0, 0))
print("Λ_z(2)=log 2:", np.log(2), "Term: 2·Λ/√n·G =", 2*np.log(2)/np.sqrt(2)*Gjk_fast(np.log(2),0,0))

# Compute prime sum slowly with finer integration
def Gjk_precise(x, j, k):
 if abs(x) >= 2*X: return 0.0
 lo = max(-X, x-X); hi = min(X, x+X)
 if lo >= hi: return 0.0
 val, err = sint.quad(lambda u: (1-np.abs(u)/X)*np.cos(j*np.pi*u/X) * (1-np.abs(x-u)/X)*np.cos(k*np.pi*(x-u)/X),
 lo, hi, limit=300)
 return val/(4*np.pi**2)

# Compare
ln2 = np.log(2)
print("Gjk_fast(log2,0,0):", Gjk_fast(ln2,0,0), "Gjk_precise:", Gjk_precise(ln2,0,0))

# Try direct prime sum with precise Gjk
prime_00 = 0.0
for n in range(2, N_TOP+1):
 if Lambda_z[n] != 0:
 ln = np.log(n)
 if ln < 2*X:
 prime_00 += Lambda_z[n]/np.sqrt(n) * Gjk_precise(ln, 0, 0)
print("Prime[0,0] precise:", 2*prime_00)
print("vs fast:", M_prime_z[0,0])


G_00(log 2): 0.06476899217123402
Λ_z(2)=log 2: 0.6931471805599453 Term: 2·Λ/√n·G = 0.06349033202010272
Gjk_fast(log2,0,0): 0.06476899217123402 Gjk_precise: 0.06476856441788573
Prime[0,0] precise: 1.2622120482510952
vs fast: 1.262206966343379


In [53]:
# Prime sum agrees between fast and precise (~1.262). So Prime[0,0]=1.262. So either Arch or Pole is wrong by ~1.13.
# 
# Let's verify with a direct independent computation using the GAUSSIAN we previously confirmed.
# For h = g_0(t)² where g_0(t) is our specific basis function (X=4):
# By explicit formula identity:
# Σ_ρ g_0(γ_ρ)² = M_arch_z[0,0] + M_pole_z[0,0] - M_prime_z[0,0]
# LHS ≈ 2 Σ_{first 5000 γ>0} g_0(γ)² ≈ M_zero_zeta[0,0] = 5.92e-6.
# 
# This should match arch+pole-prime ≈ -1.13. So we are missing 1.13 from arithmetic side. 
# But the explicit formula is an IDENTITY. So the bug is somewhere.
# 
# Let me re-check the prime sum coefficient. I wrote "Prime = 2 Σ_n Λ(n)/√n · G_jk(log n)". Let me re-derive.
# 
# Explicit formula: Σ_ρ h(γ_ρ) = pole + arch(h) - (something with primes)
# 
# Previously verified for ζ with Gaussian h:
# Σ_γ h(γ) = 2 h(i/2) + (1/(2π)) ∫ h(r)(-log π + Re ψ(1/4+ir/2)) dr - 2 Σ_n Λ(n)/√n · g(log n)
# 
# Here g(x) = FT(h)(x)/(2π) = (1/(2π)) ∫ h(r) e^{-irx} dr.
# 
# For h = g_0(t) · g_0(t) (j=k=0 case): FT(h) = FT(g_0 · g_0) = (ĝ_0 ⋆ ĝ_0)/(2π). 
# So g(x) := FT(h)(x)/(2π) = (ĝ_0 ⋆ ĝ_0)(x)/(4π²) = Gjk_fast(x, 0, 0). ✓
# 
# Then the prime term is "−2 Σ Λ(n)/√n · g(log n)" on the RHS, i.e., it gets ADDED to RHS with a MINUS sign.
# RHS = pole + arch − 2 Σ Λ(n)/√n · g(log n).
# 
# We compute: arch = -0.110, pole = 0.239, prime_sum (the 2Σ part) = 1.262.
# RHS = 0.239 + (-0.110) - 1.262 = -1.133.
# 
# But LHS = Σ_ρ h(γ_ρ) = M_zero_zeta[0,0] = 5.92e-6. 
# 
# So we're off by 1.13. Where?
# 
# Let me compute the arch INTEGRAL precisely (the issue must be in arch):
# (1/(2π)) ∫ g_0(r)² · Ω_ζ(r) dr.
# 
# We're using r_grid from -80 to 80, npts=4001. Let me compute with adaptive scipy.quad to check.

def arch_int_v(r):
 g = g_j_closed(r, 0)
 return g**2 * (sp_digamma(0.25 + 1j*r/2).real - np.log(np.pi))

# arch_00 for zeta
val, err = sint.quad(arch_int_v, -200, 200, limit=500)
val_scaled = val/(2*np.pi)
print("Arch[0,0] precise:", val_scaled, "err:", err)
print("vs M_arch_z[0,0]:", M_arch_z[0,0])


Arch[0,0] precise: -0.11020223744031563 err: 1.3784811986466335e-08
vs M_arch_z[0,0]: -0.11020225835038579


In [54]:
# Arch matches. Pole = 2 g_0(i/2)^2:
print("g_0(i/2):", g_j_closed_c(0.5j, 0))
# 2 g_0(i/2)^2 = 0.239 expected
gv = g_j_closed_c(0.5j, 0)
print("2 g_0(i/2)^2:", 2*gv**2)
# Looks right.

# So discrepancy of 1.13 comes from PRIME term.
# 
# Let me verify with the Gaussian approach: for ζ + Gaussian h, formula worked. Apply same logic to h = g_0(t)² for sanity.
# 
# Use Gaussian h(r)=exp(-r²/(2σ²)) with say σ matched to g_0 shape:
# g_0 is bandlimited in [-X,X]; effective time-width of g_0 ~ X = 4, but g_0(t) ~ 1/t² (slow decay), so it has heavy tails.
# h = g_0² also bandlimited (FT support [-2X,2X]). And h is even, non-negative.
# 
# Could it be that I have the wrong DEFINITION of "g" in the prime sum? Let me reverify with Gaussian.
# 
# Recall Gaussian test: h_g(r) = exp(-r²/(2σ²)), σ=20. Then g_g(x) = exp(-x²/2 · σ²)/√(2π)/σ. Wait earlier I had g(x)=(σ/√(2π))exp(-(σx)²/2).
# We confirmed prime term = 2 Σ Λ(n)/√n · g(log n) ≈ 0 (negligible). Match was perfect. So formula sign and definition correct.
# 
# For h = g_0(t)², we have g(x) = FT(g_0²)(x)/(2π) = Gjk_fast(x, 0, 0)/?? wait no!
# Recall: with h = g_0², its Fourier transform is FT(g_0 · g_0) = (1/(2π)) (ĝ_0 ⋆ ĝ_0) (we verified this earlier).
# But the formula's "g(x)" is FT(h)(x)/(2π), which = (ĝ_0 ⋆ ĝ_0)(x)/(4π²) = Gjk_fast(x, 0, 0). ✓
# 
# Hmm so signs/factors should be right. Let me cross-check arch+pole-prime against zero sum for the GAUSSIAN test for ζ, with N_TOP truncation:
sigma_g = 4.0 # less concentrated so we evaluate at wider primes
def h_gauss(r): return np.exp(-r**2/(2*sigma_g**2))
def g_gauss(x): return sigma_g/np.sqrt(2*np.pi)*np.exp(-(sigma_g*x)**2/2)

# Zero sum
lhs = 2*np.sum(h_gauss(zeta_g))
# Arch
arch_v, _ = sint.quad(lambda r: h_gauss(r)*(sp_digamma(0.25+1j*r/2).real - np.log(np.pi)), -200, 200, limit=500)
arch_v /= 2*np.pi
# Pole
pole_v = 2*h_gauss(0.5j.imag*1j*1j) # h(i/2) = exp(-(i/2)²/(2σ²))=exp(1/(8σ²))
pole_v = 2*np.exp(1/(8*sigma_g**2))
# Prime
prime_v = 0.0
for n in range(2, 50000):
 fac = factorint(n)
 if len(fac)==1:
 p,_=next(iter(fac.items()))
 prime_v += np.log(p)/np.sqrt(n) * g_gauss(np.log(n))
prime_v *= 2

print("LHS:", lhs)
print("Arch:", arch_v, "Pole:", pole_v, "Prime:", prime_v)
print("RHS=arch+pole-prime:", arch_v+pole_v-prime_v)
print("Diff LHS-RHS:", lhs-(arch_v+pole_v-prime_v))


g_0(i/2): (0.345735870113322+0j)
2 g_0(i/2)^2: (0.23906658376603174+0j)
LHS: 0.00388829070707166
Arch: -1.9781679020035448 Pole: 2.015686194412896 Prime: 0.03363000170228708
RHS=arch+pole-prime: 0.003888290707064257
Diff LHS-RHS: 7.402932433731024e-15


In [55]:
# Gaussian h_gauss matches perfectly. Now repeat with h = g_0(t)² but using EXACT prime sum (compute σ_n Λ(n)/√n · G_00(log n) using all n up to say 50000):
import time
t0=time.time()
prime_v = 0.0
for n in range(2, 50000):
 fac = factorint(n)
 if len(fac)==1:
 p,_=next(iter(fac.items()))
 ln = np.log(n)
 if ln >= 2*X: continue
 prime_v += np.log(p)/np.sqrt(n) * Gjk_precise(ln, 0, 0)
print(f"Direct prime sum for h=g_0²: 2× = {2*prime_v}, t={time.time()-t0:.1f}s")
# Wait, G_jk_fast/precise compute the convolution kernel only — supported on [-2X, 2X]. With X=4, log n ≤ 8 ⟹ n ≤ 2981.
# So n>2981 contribute 0 to the sum. n=2..2981 covers it.

# So our M_prime_z[0,0] should be exact prime sum: 1.262.
# Let me compute zero sum LHS now for the basis function h=g_0²:
zero_sum_g0 = 2*np.sum([g_j_closed(gam, 0)**2 for gam in zeta_g])
print("Zero sum 2·Σ g_0(γ)²:", zero_sum_g0)
print("vs M_zero_zeta[0,0]:", M_zero_zeta[0,0])

# And M_arch + M_pole - M_prime = ?
print("Arch:", M_arch_z[0,0], "Pole:", M_pole_z[0,0], "Prime:", M_prime_z[0,0])
print("Arch+Pole-Prime:", M_arch_z[0,0]+M_pole_z[0,0]-M_prime_z[0,0])


Direct prime sum for h=g_0²: 2× = 1.2622120482510952, t=0.2s
Zero sum 2·Σ g_0(γ)²: 5.919477305807289e-06
vs M_zero_zeta[0,0]: 5.919477305807289e-06
Arch: -0.11020225835038579 Pole: 0.23906658376603174 Prime: 1.262206966343379
Arch+Pole-Prime: -1.133342640927733


In [56]:
# So we have:
# Σ g_0(γ)² (truncated to 5000 zeros) = 5.92e-6
# Arch + Pole - Prime = -1.133
# Discrepancy = 1.133.
# 
# The explicit formula identity says these should be EQUAL. So we have a bug.
# 
# Hypothesis: the discrepancy comes from the prime sum NEEDING to also include n=1? Λ(1)=0 always, so no.
# OR a coefficient of 1/2: maybe the prime sum should be Σ Λ/√n · g(log n) not 2× that.
# Compare to the Gaussian case where 2× gave correct match. So 2× is right.
# 
# Let me try once more: explicit formula on h = g_0². COMPARE to Gaussian by mixing/matching coefficients.
# 
# Could it be that g_0(t) has heavy tails and the σ ≈ truncation matters?
# g_0(t) ~ 1/t² for large t. So g_0(t)² ~ 1/t⁴, integrable but heavy-tailed.
# Σ_γ g_0(γ)² over first 5000 zeros: only captures γ ≤ 5447. Heavy tail.
# Estimate tail: zeros density ~ log(γ)/(2π). Σ_{γ > T} g_0(γ)² ~ ∫_T (logγ /(2π)) · 1/γ⁴ dγ = O(log T/T³).
# For T=5447: ~log(5447)/5447³ ≈ 8.6/1.6e11 ≈ 5e-11. So tail negligible.
# 
# Therefore tail of zero sum is ~5e-11; total LHS ≈ 5.92e-6. But arch+pole-prime says -1.133. So we're off by 1.133.
# 
# Sanity check: for the Gaussian, the prime sum was 0.033 (small because g_gauss has σ=4 so g_gauss(log n) decays fast in n).
# For our basis function, prime sum is 1.26 — that's MUCH LARGER. Maybe the formula has different normalization for "g"?
# 
# Let me carefully redo the Gaussian h to make sure: σ=4, h=exp(-r²/32), g_gauss(x) = 4/√(2π) exp(-8x²).
# At n=2: g_gauss(log 2)=4/√(2π) · exp(-8·log²2)=1.596·exp(-3.84)= 1.596·0.0215 = 0.0343.
# At n=3: g_gauss(log 3)=1.596·exp(-8·log²3)=1.596·exp(-9.66)=1.596·6.4e-5= 1.02e-4.
# So prime sum ~ log(2)/√2·0.0343·2 + ... = 0.69/1.41·0.0343·2 ≈ 0.0336. Matches!

# For h=g_0(t)², the FT g(x) = G_jk_fast(x,0,0). G_00(0) = 0.0338. G_00(log 2)=0.0648, G_00(log 3)=0.0463...
# These are LARGER than the Gaussian (sigma=4) values, especially for small x. Hence prime sum is larger.
# 
# OK so let me COMPUTE the LHS Σ_γ h(γ) for h = g_0(t)² using more zeros — maybe we ARE missing significant contribution from γ ≤ 5447.

# Wait — we have 5000 ZETA zeros and used all of them. So zero sum captured. Yet diff is 1.13?
# 
# Hmm let me TRIPLE-check by an alternative path: use h = a Gaussian-shaped function with very high σ that mimics g_0:
# Actually let's just plot g_0:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
tt = np.linspace(-50, 50, 2000)
gg = [g_j_closed(t, 0) for t in tt]
print("g_0(0)=", gg[1000])
print("∫g_0²= ", np.trapezoid([g**2 for g in gg], tt))
# this should equal Σ_n? No.

# Wait — by Parseval: ∫ |g_0(t)|² dt = (1/(2π)) ∫ |ĝ_0(u)|² du = (1/(2π)) ∫_{-X}^X (1-|u|/X)² du = (1/(2π)) · (2/3)·X = X/(3π).
print("Parseval: X/(3π)=", X/(3*np.pi))


g_0(0)= 0.31824351079686874
∫g_0²= 0.2122063916271198
Parseval: X/(3π)= 0.4244131815783876


In [57]:
# Wait — Parseval says ∫|g_0|² = ∫|ĝ_0|²/(2π). ĝ_0(u)=(1-|u|/X) on [-X,X].
# ∫|ĝ_0|² = ∫_{-X}^X (1-|u|/X)² du = 2 ∫_0^X (1-u/X)² du = 2·X/3.
# Divide by 2π: X·2/(3·2π) = X/(3π) = 4/(3π) ≈ 0.4244.
# But I got 0.2122 numerically — half!
# 
# Aha—I think Parseval has a 1/(2π) factor convention mismatch.
# Convention: g(t) = (1/(2π)) ∫ ĝ(u) e^{itu} du, ĝ(u) = ∫ g(t) e^{-itu} dt.
# Then ∫ |g(t)|² dt = (1/(2π)) ∫ |ĝ(u)|² du. So with ĝ_0(u)=(1-|u|/X) (just defined ON [-X,X]):
# ∫|ĝ_0|² = 2X/3, divide by 2π: X/(3π).
# Numerical: 0.21 ≈ X/(6π)≈ 0.212. So actual numerical = X/(6π). Half of theoretical.
# 
# Let me recompute Parseval. Hmm maybe my range -50 to 50 doesn't capture full integral?
# g_0(t) ~ 1/t² so beyond ±50 contributes ∫_{50}^∞ 1/t⁴ dt ~ 50^-3/3 ~ 3e-6. Negligible.
# So numerical is 0.212. Theory says 0.424. Factor 2 off.
# 
# Likely my formula has factor of 2 error somewhere. Let me redo g_0 derivation:
# ĝ_0(u) = 1 - |u|/X on [-X,X], 0 elsewhere.
# g_0(t) = (1/(2π)) ∫_{-X}^X (1-|u|/X) e^{itu} du = (1/(2π))·2 ∫_0^X (1-u/X) cos(tu) du = (1/π) ∫_0^X (1-u/X) cos(tu) du.
# That's what I have. So formula correct.
# 
# Then Parseval check: ∫ g_0(t)² dt = (1/(2π)) ∫ ĝ_0(u)² du = (1/(2π))·(2X/3) = X/(3π) ≈ 0.424.
# But numerical gives ~0.212. So I'm getting half.
# 
# Maybe my closed form is OFF by 1/√2?? Let me recompute g_0(0):
# g_0(0) = (1/π) ∫_0^X (1-u/X) du = (1/π)·X/2 = X/(2π). For X=4: 4/(2π)≈0.6366.
# But I got 0.318. Half!
# 
# OH I see. Let me check the closed form expression.
def g_j_closed(r, j, X=X):
 a = j*np.pi/X
 out = 0.0
 for sign in [1,-1]:
 c = a + sign*r
 if abs(c) < 1e-12: out += X/2
 else: out += (1 - np.cos(c*X))/(X*c**2)
 return out/(2*np.pi)

# At r=0, j=0: a=0, c=0+sign·0=0 → both signs trigger "abs(c)<1e-12" giving X/2 each: total = X. Divided by 2π → X/(2π) = 0.6366.
# But I checked earlier and got 0.318. Let me re-check:
X = 4.0
print("g_0(0):", g_j_closed(0.0, 0, X=4))
print("expected X/(2π):", X/(2*np.pi))


g_0(0): 0.6366197723675814
expected X/(2π): 0.6366197723675814


In [58]:
# Wait, but the precomputed gj_grid uses X=4 (current), but earlier when X=2 (in the matrix construction) we had g_0(0)=0.318.
# Actually I see — the run cell #45 with J=10 hadn't reset X correctly inside the function closure? Let me check what X is "see" inside g_j_closed:
# g_j_closed takes X as default arg, set at function definition (X=2 when first defined). Subsequent reassignment of global X doesn't change default.
# 
# THIS IS THE BUG. The closed-form g_j_closed function has X=2 baked in. So all my J=10 computations actually used X=2! 

# Re-define with current X
X = 4.0
def g_j_closed(r, j, X=X):
 a = j*np.pi/X
 out = 0.0
 for sign in [1,-1]:
 c = a + sign*r
 if abs(c) < 1e-12: out += X/2
 else: out += (1 - np.cos(c*X))/(X*c**2)
 return out/(2*np.pi)

def g_j_closed_c(z, j, X=X):
 a = j*np.pi/X
 out = 0.0+0.0j
 for sign in [1,-1]:
 c = a + sign*z
 if abs(c) < 1e-12: out += X/2
 else: out += (1 - np.cos(c*X))/(X*c**2)
 return out/(2*np.pi)

def Gjk_fast(x, j, k, X=X):
 if abs(x) >= 2*X: return 0.0
 lo = max(-X, x-X); hi = min(X, x+X)
 nint = 401
 us = np.linspace(lo, hi, nint)
 v1 = (1 - np.abs(us)/X) * np.cos(j*np.pi*us/X)
 v2 = (np.abs(x-us)<=X) * (1 - np.abs(np.clip(x-us,-X,X))/X) * np.cos(k*np.pi*(x-us)/X)
 return np.trapezoid(v1*v2, us)/(4*np.pi**2)

print("g_0(0)=", g_j_closed(0, 0))
# Re-do gj_grid:
R_int = 80.0
npts = 4001
r_grid = np.linspace(-R_int, R_int, npts)
J = 10
gj_grid = np.zeros((J, npts))
for j in range(J):
 for i, r in enumerate(r_grid):
 gj_grid[j, i] = g_j_closed(r, j)

# Re-do G_arr
N_TOP = int(np.exp(2*X))+1
n_arr = np.arange(2, N_TOP+1)
log_n_arr = np.log(n_arr)
sqrt_n_arr = np.sqrt(n_arr)
print("N_TOP:", N_TOP, "n in [2,", N_TOP, "]")

G_arr = np.zeros((J, J, len(n_arr)))
for j in range(J):
 for k in range(j, J):
 for i, n in enumerate(n_arr):
 ln = log_n_arr[i]
 if ln >= 2*X: continue
 G_arr[j,k,i] = Gjk_fast(ln, j, k)
 G_arr[k,j] = G_arr[j,k]
print("G_arr done")


g_0(0)= 0.6366197723675814
N_TOP: 2981 n in [2, 2981 ]


G_arr done


In [59]:
def compute_arch_matrix(Omega_L):
 omega_vals = np.array([Omega_L(r) for r in r_grid])
 M = np.zeros((J,J))
 for j in range(J):
 for k in range(j,J):
 integrand = gj_grid[j] * gj_grid[k] * omega_vals
 v = np.trapezoid(integrand, r_grid)/(2*np.pi)
 M[j,k] = v; M[k,j] = v
 return M

def prime_matrix(Lambda_arr):
 M = np.zeros((J,J))
 weights = Lambda_arr[2:N_TOP+1] / sqrt_n_arr
 for j in range(J):
 for k in range(j,J):
 M[j,k] = 2*np.sum(weights * G_arr[j,k])
 M[k,j] = M[j,k]
 return M

def compute_pole_matrix(z_pole=None, factor=1.0):
 M = np.zeros((J,J))
 if z_pole is None: return M
 g_vec = np.array([g_j_closed_c(z_pole, j).real for j in range(J)])
 return factor * np.outer(g_vec, g_vec)

# Rebuild Lambda arrays for n up to N_TOP
Lambda_z = np.zeros(N_TOP+1)
for n in range(2, N_TOP+1):
 fac = factorint(n)
 if len(fac)==1:
 p, _ = next(iter(fac.items()))
 Lambda_z[n] = np.log(p)

Lambda_chi5 = np.zeros(N_TOP+1)
for n in range(2, N_TOP+1):
 fac = factorint(n)
 if len(fac)==1:
 p, k = next(iter(fac.items()))
 cv = chi5val(p)**k
 Lambda_chi5[n] = np.log(p)*cv

a_DH_arr2 = np.zeros(N_TOP+1)
for n in range(1, N_TOP+1):
 r = n%5
 a_DH_arr2[n] = {0:0.0,1:1.0,2:alpha_f,3:-alpha_f,4:-1.0}[r]
Lambda_DH_arr2 = np.zeros(N_TOP+1)
for n in range(2, N_TOP+1):
 val = a_DH_arr2[n]*np.log(n)
 for d in range(2,n+1):
 if n%d==0 and d<n:
 val -= a_DH_arr2[d]*Lambda_DH_arr2[n//d]
 Lambda_DH_arr2[n] = val

print("Arithmetic data done.")

M_arch_z = compute_arch_matrix(Omega_zeta)
M_arch_chi = compute_arch_matrix(Omega_chi5)
M_arch_DH = compute_arch_matrix(Omega_DH)
M_prime_z = prime_matrix(Lambda_z)
M_prime_chi = prime_matrix(Lambda_chi5)
M_prime_DH = prime_matrix(Lambda_DH_arr2)
M_pole_z = compute_pole_matrix(z_pole=0.5j, factor=2.0)

M_z = M_arch_z + M_pole_z - M_prime_z
M_chi = M_arch_chi - M_prime_chi
M_DH = M_arch_DH - M_prime_DH

print("\nEigenvalues:")
print("ζ: ", np.linalg.eigvalsh(M_z))
print("L(χ_5):", np.linalg.eigvalsh(M_chi))
print("L_DH: ", np.linalg.eigvalsh(M_DH))


Arithmetic data done.

Eigenvalues:
ζ: [-5.35641291e-06 -5.00266219e-07 -1.92630581e-12 -1.07104385e-12
 2.05991453e-13 3.42016154e-11 1.06393330e-08 4.96127080e-07
 8.73947840e-06 1.71253288e-05]
L(χ_5): [-2.01562475e-07 -4.76333614e-08 -2.27234531e-13 7.84252224e-12
 1.63163930e-09 3.49256475e-07 1.46338194e-06 1.17861625e-05
 3.41597722e-05 2.69296646e-01]
L_DH: [-3.39534162e-07 -1.96815325e-07 1.46107204e-14 4.56465634e-11
 7.53349558e-09 1.09641297e-07 2.15212329e-06 7.80605609e-06
 2.52460382e-04 2.71123965e-01]


In [60]:
# Now eigenvalues are much smaller. λ_min:
# ζ: -5.4e-6
# L(χ_5): -2.0e-7
# L_DH: -3.4e-7
#
# All NEGATIVE but very small. Likely all due to numerical integration error (we'd need higher precision).
# Verify by checking explicit formula identity (LHS-zero-sum vs RHS-arithmetic):

# Build zero-sum matrix
gj_at_zeta_new = np.zeros((J, len(zeta_g)))
for j in range(J):
 for i, gam in enumerate(zeta_g):
 gj_at_zeta_new[j, i] = g_j_closed(gam, j)
M_zero_z = np.zeros((J,J))
for j in range(J):
 for k in range(j,J):
 M_zero_z[j,k] = 2*np.sum(gj_at_zeta_new[j]*gj_at_zeta_new[k])
 M_zero_z[k,j] = M_zero_z[j,k]
print("M_zero_z[0,0]:", M_zero_z[0,0], "vs M_z[0,0]=arch+pole-prime:", M_z[0,0])
print("Identity diff (max):", np.max(np.abs(M_z - M_zero_z)))
print("Identity diff (Frobenius):", np.linalg.norm(M_z - M_zero_z))


M_zero_z[0,0]: 2.7896017647278743e-07 vs M_z[0,0]=arch+pole-prime: 5.3562825692576865e-06
Identity diff (max): 5.077322392784899e-06
Identity diff (Frobenius): 1.3614034641762875e-05


In [61]:
# Identity holds to ~5e-6. So most negative eigenvalues are numerical noise.
# Let me improve the arch integration precision by using scipy.quad (adaptive) instead of trapezoid.
import scipy.integrate as sint

def compute_arch_matrix_quad(Omega_L, R=200):
 M = np.zeros((J,J))
 for j in range(J):
 for k in range(j,J):
 def integrand(r):
 return g_j_closed(r, j) * g_j_closed(r, k) * Omega_L(r)
 v, err = sint.quad(integrand, -R, R, limit=500)
 M[j,k] = v/(2*np.pi); M[k,j] = M[j,k]
 return M

import time
t0=time.time()
M_arch_z_q = compute_arch_matrix_quad(Omega_zeta)
print(f"Arch ζ quad ({time.time()-t0:.1f}s)")
print("diff:", np.linalg.norm(M_arch_z - M_arch_z_q))
print("eigs ζ (with quad arch):", np.linalg.eigvalsh(M_arch_z_q + M_pole_z - M_prime_z))


Arch ζ quad (0.8s)
diff: 3.873840406533804e-08
eigs ζ (with quad arch): [-5.35162434e-06 -4.97831309e-07 -4.56563635e-13 3.10766572e-12
 6.74092655e-12 5.85709621e-11 1.06403864e-08 4.96875577e-07
 8.76724199e-06 1.71417514e-05]


In [62]:
# Arch precision doesn't help much. The issue is probably the prime sum integration (G_jk) precision. Use Gjk_precise:
import time
t0 = time.time()
G_arr_precise = np.zeros((J, J, len(n_arr)))
for j in range(J):
 for k in range(j, J):
 for i, n in enumerate(n_arr):
 ln = log_n_arr[i]
 if ln >= 2*X: continue
 v, _ = sint.quad(lambda u: (1-np.abs(u)/X)*np.cos(j*np.pi*u/X) * 
 (np.abs(ln-u)<=X)*(1-np.abs(np.clip(ln-u,-X,X))/X)*np.cos(k*np.pi*(ln-u)/X),
 max(-X, ln-X), min(X, ln+X), limit=200)
 G_arr_precise[j,k,i] = v/(4*np.pi**2)
 G_arr_precise[k,j] = G_arr_precise[j,k]
print(f"G precise done ({time.time()-t0:.1f}s)")
diff_G = np.linalg.norm(G_arr_precise - G_arr)
print("G diff norm:", diff_G)

def prime_matrix_precise(Lambda_arr):
 M = np.zeros((J,J))
 weights = Lambda_arr[2:N_TOP+1] / sqrt_n_arr
 for j in range(J):
 for k in range(j,J):
 M[j,k] = 2*np.sum(weights * G_arr_precise[j,k])
 M[k,j] = M[j,k]
 return M

M_prime_z_p = prime_matrix_precise(Lambda_z)
M_prime_chi_p = prime_matrix_precise(Lambda_chi5)
M_prime_DH_p = prime_matrix_precise(Lambda_DH_arr2)

M_z_p = M_arch_z_q + M_pole_z - M_prime_z_p
M_chi_p = compute_arch_matrix_quad(Omega_chi5) - M_prime_chi_p
M_DH_p = compute_arch_matrix_quad(Omega_DH) - M_prime_DH_p

print()
print("Eigenvalues with high-precision quadrature:")
print("ζ: ", np.linalg.eigvalsh(M_z_p))
print("L(χ_5):", np.linalg.eigvalsh(M_chi_p))
print("L_DH: ", np.linalg.eigvalsh(M_DH_p))


G precise done (33.9s)
G diff norm: 6.926595808223246e-06



Eigenvalues with high-precision quadrature:
ζ: [-6.50884493e-10 -2.16790673e-10 -1.42534114e-10 2.97417221e-11
 2.99750278e-10 7.32614657e-10 1.96611924e-09 6.10691227e-08
 1.66247479e-06 1.55816563e-05]
L(χ_5): [-7.98492822e-10 -2.02147706e-10 6.92360372e-11 3.72985026e-10
 1.72985617e-09 5.93207641e-08 7.85663414e-07 1.24999428e-05
 3.51074800e-05 2.69296746e-01]
L_DH: [-1.72256284e-10 7.31956295e-11 8.55935344e-11 4.48153569e-10
 3.02378706e-09 3.00351294e-08 1.91881850e-06 7.51321661e-06
 2.53228648e-04 2.71123289e-01]


In [63]:
# With high-precision quadrature:
# ζ: λ_min = -6.5e-10 (essentially zero, numerical)
# χ_5: λ_min = -8.0e-10 (essentially zero)
# L_DH: λ_min = -1.7e-10 (essentially zero)
#
# All three matrices are NUMERICALLY PSD (zero λ_min within numerical precision).
# The L_DH matrix does NOT show negative λ_min as predicted by the hypothesis.
# 
# Why? Because the off-line zeros of L_DH have IMAGINARY shift |β-1/2| ≤ 0.4 (max 0.398, see CSV), 
# and our test function basis g_j with X=4 is bandlimited to [-4,4] in frequency. 
# At a complex point γ - i·δ with δ < 0.5, g_j(γ-iδ) = (1/(2π))∫ĝ_j(u) e^{i(γ-iδ)u}du = (1/(2π))∫ĝ_j(u) e^{δu} e^{iγu} du.
# The shift in u introduces a factor e^{δu} which is bounded by e^{Xδ} = e^{4·0.5} = e^2 ≈ 7.4.
# So |g_j(γ-iδ)| is at most ~7.4 |g_j(γ)|, manageable.
# 
# The off-line contribution to the matrix Σ_ρ g_j(τ)g_k(τ) Re part is:
# For each off-line ρ=σ+it: 4 Re[g_j(t-i(σ-1/2)) g_k(t-i(σ-1/2))]. 
# 
# Compute this contribution:
off_contrib = np.zeros((J,J))
for _, row in offline.iterrows():
 sig=row['sigma']; tt=row['t']
 z = complex(tt, -(sig-0.5))
 gvec = np.array([g_j_closed_c(z, j) for j in range(J)])
 # Contribution: 4 Re[outer(gvec, gvec)] for the 4 partners
 out_g = np.outer(gvec, gvec)
 off_contrib += 4*np.real(out_g)

print("Off-line contribution matrix (Frobenius):", np.linalg.norm(off_contrib))
print("Off-line eigenvalues:", np.linalg.eigvalsh(off_contrib))
print()
print("M_DH eigenvalues:", np.linalg.eigvalsh(M_DH_p))
print("M_DH - off_contrib eigenvalues (i.e., critical-line-only part):", np.linalg.eigvalsh(M_DH_p - off_contrib))


Off-line contribution matrix (Frobenius): 2.09474429082354e-09
Off-line eigenvalues: [-4.97043418e-15 -9.51993996e-26 -1.21500786e-26 1.34909342e-25
 2.48917684e-25 9.74997808e-25 3.54714338e-22 7.27287674e-16
 5.28920346e-10 2.02686875e-09]

M_DH eigenvalues: [-1.72256284e-10 7.31956295e-11 8.55935344e-11 4.48153569e-10
 3.02378706e-09 3.00351294e-08 1.91881850e-06 7.51321661e-06
 2.53228648e-04 2.71123289e-01]
M_DH - off_contrib eigenvalues (i.e., critical-line-only part): [-1.72284889e-10 7.31914768e-11 8.55903584e-11 4.48114882e-10
 3.02119224e-09 3.00197474e-08 1.91807538e-06 7.51272338e-06
 2.53227894e-04 2.71123289e-01]


In [64]:
# Off-line contribution is TINY (Frobenius ~ 2e-9). It has λ_min = -5e-15 essentially zero.
# So the 110 off-line zeros barely contribute. They are at t ≥ 114 where g_j(t) is already small (g_j(t) ~ 1/t² gives 1/114² ≈ 8e-5).
# 
# Net result: ALL FIVE (well, three available) L-functions show λ_min ≈ 0 within numerical precision (~1e-10).
# No negative λ_min is observed for L_DH, REFUTING the specific quantitative hypothesis at this basis size.
# 
# To possibly see negative eigenvalues, we would need test functions concentrated near specific off-line zero positions, 
# OR we would need a much higher J to span the relevant test-function space.
# 
# Let me also try a more discriminating test: increase J to capture more "resolution" near off-line zeros.
# Also try test functions specifically centered at off-line zero locations: use modulated bumps.

# Try alternative basis: shifted Hermite-like or sinc-modulated bumps centered at γ ~ off-line zero heights (e.g. 114, 166, 240, ...).
# 
# Actually let's also try ENRICHING the basis with frequencies that "see" the off-line region.
# The off-line zeros are at γ heights of ~114 to ~5000. To "feel" a zero at γ=114, we need test function g(t) with 
# significant value at t=114. Our g_j(t) = (1-cos(c X))/(X c²)/(2π) where c = jπ/X ± t. For t=114, this is ~1/(2π·114²) ≈ 1.2e-5.
# 
# To make a test function PEAKED at γ=114, use ĝ(u) = e^{i·114·u} · bump(u). But this is not real-symmetric.
# Use COSINE modulation: ĝ(u) = cos(114·u) · (1-|u|/X). Then g(t) is the Fourier transform: peaks at t=±114.
# 
# Let me add such localized test functions.
X = 4.0
# Augment basis: g_extra(t) with cos(t0·u) modulation, t0=γ off-line zero
extra_centers = [114.16, 166.48, 240.40, 357.79] # first few off-line zero heights

# Build augmented basis
def g_extra(t, t0, X=X):
 # g(t) = (1/(2π)) ∫_{-X}^X cos(t0·u)(1-|u|/X) e^{itu} du
 # = (1/(2π))·2 ∫_0^X cos(t0·u)(1-u/X) cos(t·u) du
 # = (1/π) ∫_0^X (1-u/X)[cos((t0+t)u)+cos((t0-t)u)]/2 du
 # = (1/(2π)) [I(t0+t) + I(t0-t)] where I(c)=(1-cos(cX))/(Xc²)
 res = 0.0
 for sign in [1,-1]:
 c = t0 + sign*t
 if abs(c) < 1e-12: res += X/2
 else: res += (1-np.cos(c*X))/(X*c**2)
 return res/(2*np.pi)

def g_extra_c(z, t0, X=X):
 res = 0.0+0.0j
 for sign in [1,-1]:
 c = t0 + sign*z
 if abs(c) < 1e-12: res += X/2
 else: res += (1-np.cos(c*X))/(X*c**2)
 return res/(2*np.pi)

# Note: g_extra(t, t0=jπ/X) reproduces our g_j (with same form, since c = jπ/X + sign·t).
# So g_extra is a generalization. Let me build a richer basis indexed by t0:
basis_t0 = [j*np.pi/X for j in range(10)] + extra_centers
J_aug = len(basis_t0)
print("J_aug:", J_aug)
print("basis_t0:", basis_t0)


J_aug: 14
basis_t0: [0.0, 0.7853981633974483, 1.5707963267948966, 2.356194490192345, 3.141592653589793, 3.9269908169872414, 4.71238898038469, 5.497787143782138, 6.283185307179586, 7.0685834705770345, 114.16, 166.48, 240.4, 357.79]


In [65]:
# Build augmented matrix. ĝ_j(u) = cos(t0_j · u) (1-|u|/X) on [-X,X].
# G_jk(x) = (1/(4π²)) ∫_{-X}^X (1-|u|/X)cos(t0_j u) (1-|x-u|/X)cos(t0_k (x-u)) · 1_{|x-u|≤X} du
# (for |x|<2X; else 0)

import scipy.integrate as sint

def ghat_general(u, t0):
 if abs(u) > X: return 0.0
 return (1-abs(u)/X)*np.cos(t0*u)

def Gjk_general(x, t0_j, t0_k):
 if abs(x) >= 2*X: return 0.0
 lo = max(-X, x-X); hi = min(X, x+X)
 val, _ = sint.quad(lambda u: ghat_general(u, t0_j)*ghat_general(x-u, t0_k),
 lo, hi, limit=300)
 return val/(4*np.pi**2)

# Build G_aug
J = J_aug
print("Building G_aug...")
import time; t0_=time.time()
G_aug = np.zeros((J, J, len(n_arr)))
for jj in range(J):
 for kk in range(jj, J):
 for i, n in enumerate(n_arr):
 ln = log_n_arr[i]
 if ln >= 2*X: continue
 G_aug[jj,kk,i] = Gjk_general(ln, basis_t0[jj], basis_t0[kk])
 G_aug[kk,jj] = G_aug[jj,kk]
print(f"G_aug done ({time.time()-t0_:.1f}s)")

# Build arch matrix
print("Building arch matrices...")
gj_aug_grid = np.zeros((J, npts))
for jj in range(J):
 for i, r in enumerate(r_grid):
 gj_aug_grid[jj, i] = g_extra(r, basis_t0[jj])

def compute_arch_aug(Omega_L):
 M = np.zeros((J, J))
 omega = np.array([Omega_L(r) for r in r_grid])
 for j in range(J):
 for k in range(j, J):
 v = np.trapezoid(gj_aug_grid[j]*gj_aug_grid[k]*omega, r_grid)/(2*np.pi)
 M[j,k]=v; M[k,j]=v
 return M

t0_=time.time()
M_arch_z_aug = compute_arch_aug(Omega_zeta)
M_arch_chi_aug = compute_arch_aug(Omega_chi5)
M_arch_DH_aug = compute_arch_aug(Omega_DH)
print(f"Arch done ({time.time()-t0_:.1f}s)")

def prime_aug(Lambda_arr):
 weights = Lambda_arr[2:N_TOP+1] / sqrt_n_arr
 M = np.zeros((J,J))
 for j in range(J):
 for k in range(j,J):
 M[j,k] = 2*np.sum(weights * G_aug[j,k])
 M[k,j] = M[j,k]
 return M

M_prime_z_aug = prime_aug(Lambda_z)
M_prime_chi_aug = prime_aug(Lambda_chi5)
M_prime_DH_aug = prime_aug(Lambda_DH_arr2)

# Pole
g_vec_pole = np.array([g_extra_c(0.5j, t0).real for t0 in basis_t0])
M_pole_aug = 2 * np.outer(g_vec_pole, g_vec_pole)

M_z_aug = M_arch_z_aug + M_pole_aug - M_prime_z_aug
M_chi_aug = M_arch_chi_aug - M_prime_chi_aug
M_DH_aug = M_arch_DH_aug - M_prime_DH_aug

print("\nAugmented basis eigenvalues:")
print("ζ: ", np.linalg.eigvalsh(M_z_aug))
print("L(χ_5):", np.linalg.eigvalsh(M_chi_aug))
print("L_DH: ", np.linalg.eigvalsh(M_DH_aug))


Building G_aug...


/tmp/ipykernel_20/3448841316.py:14: IntegrationWarning: The integral is probably divergent, or slowly convergent.
 val, _ = sint.quad(lambda u: ghat_general(u, t0_j)*ghat_general(x-u, t0_k),


/tmp/ipykernel_20/3448841316.py:14: IntegrationWarning: The maximum number of subdivisions (300) has been achieved.
 If increasing the limit yields no improvement it is advised to analyze 
 the integrand in order to determine the difficulties. If the position of a 
 local difficulty can be determined (singularity, discontinuity) one will 
 probably gain from splitting up the interval and calling the integrator 
 on the subranges. Perhaps a special-purpose integrator should be used.
 val, _ = sint.quad(lambda u: ghat_general(u, t0_j)*ghat_general(x-u, t0_k),


/tmp/ipykernel_20/3448841316.py:14: IntegrationWarning: The occurrence of roundoff error is detected, which prevents 
 the requested tolerance from being achieved. The error may be 
 underestimated.
 val, _ = sint.quad(lambda u: ghat_general(u, t0_j)*ghat_general(x-u, t0_k),


G_aug done (153.9s)
Building arch matrices...
Arch done (0.0s)

Augmented basis eigenvalues:
ζ: [-4.87284017e-02 -3.69152101e-02 -6.60211672e-10 -2.34793296e-10
 -1.79536666e-10 1.61068360e-11 2.88218709e-10 6.82012796e-10
 1.80038874e-09 5.92598455e-08 1.63806691e-06 1.55561662e-05
 9.20593099e-02 1.16597809e-01]
L(χ_5): [-5.26914531e-02 -2.33400632e-02 -8.04094170e-10 -1.90594578e-10
 6.90152321e-11 3.77009716e-10 1.68293210e-09 5.78942335e-08
 7.79422357e-07 1.24755767e-05 3.50702488e-05 1.23806712e-02
 4.80684905e-02 2.69296732e-01]
L_DH: [-1.74882478e-10 7.25486199e-11 8.18677441e-11 4.42211507e-10
 2.95509800e-09 2.91978455e-08 1.89797966e-06 7.48619851e-06
 2.53208400e-04 6.95360723e-02 2.46840260e-01 2.71123256e-01
 2.78519681e-01 4.02619796e-01]


In [66]:
# Striking observation:
# ζ: λ_min = -4.87e-2 (significantly negative!)
# χ_5: λ_min = -5.27e-2 (significantly negative!)
# L_DH: λ_min = -1.75e-10 (essentially zero, smallest in magnitude)
# 
# But this is the OPPOSITE of the hypothesis! L_DH has the LEAST negative λ_min.
# 
# The negative values for ζ and χ_5 come from the high-frequency basis functions (t0=114, 166, etc) which oscillate rapidly. The convolution integrals for G_jk at high t0 are problematic (numerical warnings appeared during G_aug computation).
# 
# This makes the augmented basis numerically unreliable. The TRUE answer for ζ and χ_5 should be λ_min ≥ 0 (under GRH).
# 
# Let me investigate the negative eigenvalues by looking at the eigenvectors and whether numerical instability is the cause.

# Check identity: M_aug should equal Σ_ρ g_aug_j(γ_ρ) g_aug_k(γ_ρ) over all known zeros for L_DH.
# For ζ: only critical-line zeros, identity matrix should equal Σ_ρ g(γ)g(γ).
print("Computing zero-side matrix for ζ...")
gz_at_zeros = np.zeros((J, len(zeta_g)))
for j in range(J):
 for i, gam in enumerate(zeta_g):
 gz_at_zeros[j,i] = g_extra(gam, basis_t0[j])
M_zero_z_aug = np.zeros((J,J))
for j in range(J):
 for k in range(j,J):
 M_zero_z_aug[j,k] = 2*np.sum(gz_at_zeros[j]*gz_at_zeros[k])
 M_zero_z_aug[k,j] = M_zero_z_aug[j,k]
print("Eigenvalues of M_zero_z_aug (should be PSD):")
print(np.linalg.eigvalsh(M_zero_z_aug))
print("Identity check: M_z_aug - M_zero_z_aug Frobenius:", np.linalg.norm(M_z_aug - M_zero_z_aug))


Computing zero-side matrix for ζ...
Eigenvalues of M_zero_z_aug (should be PSD):
[-8.87182661e-18 -1.12646825e-18 2.31655907e-17 3.41541851e-16
 9.28140105e-14 9.75671760e-12 1.67818889e-09 6.12206297e-08
 1.66375233e-06 1.55827343e-05 6.20808368e-02 8.61707102e-02
 1.89994136e-01 2.52981024e-01]
Identity check: M_z_aug - M_zero_z_aug Frobenius: 0.23594724808914394


In [67]:
# The zero-sum matrix has eigenvalues all ≥ 0 (PSD as expected from Gram).
# But the arithmetic-side M_z_aug differs from M_zero_z_aug by Frobenius 0.236 — meaning our arithmetic side computation has substantial error for the high-frequency basis functions.
# 
# So the negative eigenvalues we see for ζ and χ_5 are NUMERICAL ARTIFACTS due to the difficulty of integrating
# (ĝ_j ⋆ ĝ_k)(x) accurately when t0_j is large.
# 
# This confirms: with PROPER computation, the Weil matrix should be PSD for ζ, χ_5, and (with off-line zeros included) L_DH too — since the off-line contribution is too small to matter at the basis sizes we've explored.
# 
# **CONCLUSION**: We have implemented the rigorous Weil explicit formula. λ_min for all three L-functions is non-negative within numerical precision. The hypothesis that L_DH yields a negative minimum eigenvalue is NOT REPRODUCED at this resolution.
# 
# **The missing lemma**: For the Weil matrix to detect off-line zeros, the basis must include test functions whose Fourier transforms are CONCENTRATED near the imaginary heights of the off-line zeros (γ ≈ 114, 166, etc.) AND can resolve the small imaginary shifts (β-1/2 ≈ 0.05-0.4). Our standard low-frequency Paley-Wiener basis (j π/X for j=0..9, X=4) cannot probe these regions. The localized basis attempt failed due to numerical integration instability — proper treatment requires higher precision integration (mpmath) or analytic closed forms.
# 
# Given the time we've spent, let me record the rigorous low-frequency result (J=10, X=4) as the main finding and produce the figure.

# Final rigorous results: use the J=10 basis (X=4) with high-precision quadrature (M_z_p, M_chi_p, M_DH_p).
eig_z = np.linalg.eigvalsh(M_z_p)
eig_chi = np.linalg.eigvalsh(M_chi_p)
eig_DH = np.linalg.eigvalsh(M_DH_p)

print("FINAL RESULTS (J=10, X=4, high-precision quadrature):")
print(f" ζ: λ_min = {eig_z[0]:.3e}, all eigenvalues consistent with PSD (numerical floor ~1e-10)")
print(f" L(χ_5): λ_min = {eig_chi[0]:.3e}, all eigenvalues consistent with PSD")
print(f" L_DH: λ_min = {eig_DH[0]:.3e}, all eigenvalues consistent with PSD")
print()
print("L(Δ,s) and ζ_δ: zero data not available in workspace -- cannot test.")


FINAL RESULTS (J=10, X=4, high-precision quadrature):
 ζ: λ_min = -6.509e-10, all eigenvalues consistent with PSD (numerical floor ~1e-10)
 L(χ_5): λ_min = -7.985e-10, all eigenvalues consistent with PSD
 L_DH: λ_min = -1.723e-10, all eigenvalues consistent with PSD

L(Δ,s) and ζ_δ: zero data not available in workspace -- cannot test.


In [68]:
# Build final summary figure: single panel showing eigenvalue spectra (log-scale of |eigenvalues|) for all 3 L-functions, 
# highlighting λ_min.

import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 11})

fig, ax = plt.subplots(figsize=(8, 5))

j_idx = np.arange(1, J+1)
labels = ['ζ (Riemann)', 'L(χ_5) [Dirichlet, real quadratic]', 'L_DH (with 110 off-line zeros)']
eigs_list = [eig_z, eig_chi, eig_DH]
markers = ['o', 's', '^']
colors = ['#1f77b4', '#2ca02c', '#d62728']

# Plot sorted eigenvalues; use signed-log scale: sign(λ) · log10(|λ|+ε)
for eigs, lab, mk, c in zip(eigs_list, labels, markers, colors):
 ax.plot(j_idx, eigs, marker=mk, linestyle='-', color=c, label=lab, markersize=8, alpha=0.85)

ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
ax.set_xlabel('Eigenvalue index (sorted ascending)')
ax.set_ylabel(r'Eigenvalue $\lambda_i$ of Weil matrix $M$')
ax.set_yscale('symlog', linthresh=1e-10)
ax.set_title("Weil quadratic-form matrix eigenvalues\n(Paley-Wiener basis: $J=10$, support $X=4$)")
ax.legend(loc='upper left', fontsize=9, frameon=True)
ax.grid(True, alpha=0.3)

# Annotate λ_min
for eigs, c, off in zip(eigs_list, colors, [-0.55, -0.35, -0.15]):
 ax.annotate(f"λ_min = {eigs[0]:.2e}", xy=(1, eigs[0]), xytext=(2.5, eigs[0]*5 if eigs[0]<0 else 1e-9),
 color=c, fontsize=9, arrowprops=dict(arrowstyle='->', color=c, alpha=0.6))

plt.tight_layout()
plt.savefig('weil_positivity_eigenvalues.png', dpi=140, bbox_inches='tight')
plt.show()
print("Saved weil_positivity_eigenvalues.png")


ValueError: x and y must have same first dimension, but have shapes (14,) and (10,)

In [69]:
# J got reset to 14 by the augmented basis. Reset J=10 for plotting:
J = 10
j_idx = np.arange(1, J+1)

import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 11})

fig, ax = plt.subplots(figsize=(8.5, 5.5))

labels = [r'$\zeta$ (Riemann)', r'$L(\chi_5)$ [real quadratic Dirichlet]', r'$L_{DH}$ (with 110 off-line zeros)']
eigs_list = [eig_z, eig_chi, eig_DH]
markers = ['o', 's', '^']
colors = ['#1f77b4', '#2ca02c', '#d62728']

for eigs, lab, mk, c in zip(eigs_list, labels, markers, colors):
 ax.plot(j_idx, eigs, marker=mk, linestyle='-', color=c, label=lab, markersize=8, alpha=0.85)

ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
ax.set_xlabel('Eigenvalue index (sorted ascending)')
ax.set_ylabel(r'Eigenvalue $\lambda_i$ of Weil matrix $M$')
ax.set_yscale('symlog', linthresh=1e-11)
ax.set_title(r"Weil quadratic-form matrix eigenvalues" + "\n" + 
 r"(Paley-Wiener basis: $J=10$, support half-width $X=4$)" + "\n" +
 r"All $\lambda_{\min} \approx 0$ within numerical precision: $|\lambda_{\min}| < 10^{-9}$")
ax.legend(loc='upper left', fontsize=10, frameon=True)
ax.grid(True, alpha=0.3, which='both')

# Annotate λ_min with text box
text = (
 r"$\bf{\lambda_{\min}}$ values (numerical):" + "\n"
 f" ζ: {eig_z[0]:+.2e}\n"
 f" L(χ_5): {eig_chi[0]:+.2e}\n"
 f" L_DH: {eig_DH[0]:+.2e}\n\n"
 "All ≈ 0 within numerical floor.\n"
 "Off-line contribution (110 zeros)\n"
 "to M_DH is $\\|·\\|_F \\approx 2\\!\\times\\!10^{-9}$\n"
 "— below numerical precision."
)
ax.text(0.97, 0.03, text, transform=ax.transAxes, fontsize=9,
 verticalalignment='bottom', horizontalalignment='right',
 bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.95, edgecolor='gray'))

plt.tight_layout()
plt.savefig('weil_positivity_eigenvalues.png', dpi=140, bbox_inches='tight')
plt.show()
print("Saved weil_positivity_eigenvalues.png")


Saved weil_positivity_eigenvalues.png


In [70]:
# Save a CSV of results for artifact:
import pandas as pd
results = pd.DataFrame({
 'eigenvalue_index': np.arange(1, J+1),
 'zeta_eig': eig_z,
 'L_chi5_eig': eig_chi,
 'L_DH_eig': eig_DH,
})
results.to_csv('weil_matrix_eigenvalues.csv', index=False)

# Save summary
with open('weil_positivity_results.txt', 'w') as f:
 f.write("Weil Positivity Test Results\n")
 f.write("="*60+"\n\n")
 f.write("Test functions: Paley-Wiener basis with compactly supported Fourier transforms\n")
 f.write(f" ĝ_j(u) = (1-|u|/X) · cos(jπu/X) on [-X,X], j=0..{J-1}, X=4\n\n")
 f.write("Method: Computed Weil matrix M_jk = Arch_jk + Pole_jk - Prime_jk\n")
 f.write(" using arithmetic side of explicit formula (no zero list needed for ζ, L(χ_5), L_DH\n")
 f.write(" matrix construction itself).\n\n")
 f.write("L-functions tested:\n")
 f.write(" ζ: conductor 1, gamma factor Γ(s/2), pole at s=1 (Λ_ζ on prime powers)\n")
 f.write(" L(χ_5): conductor 5, gamma factor Γ(s/2) (real quadratic char, EVEN)\n")
 f.write(" L_DH: conductor 5, gamma factor Γ((s+1)/2) (no Euler product, 110 off-line zeros)\n\n")
 f.write("Minimum eigenvalues:\n")
 f.write(f" λ_min(M_ζ) = {eig_z[0]:+.3e}\n")
 f.write(f" λ_min(M_χ_5) = {eig_chi[0]:+.3e}\n")
 f.write(f" λ_min(M_DH) = {eig_DH[0]:+.3e}\n\n")
 f.write("CONCLUSION: All three matrices have minimum eigenvalue ≈ 0 within numerical\n")
 f.write("precision (~1e-10 from quadrature error). NO function shows a negative λ_min\n")
 f.write("at this basis size and choice. The hypothesis (M_DH should have negative λ_min)\n")
 f.write("is NOT reproduced.\n\n")
 f.write("Off-line contribution to M_DH (from 110 off-line zeros at t≥114):\n")
 f.write(f" ||Σ_{{off-line ρ}} g_j(τ_ρ)g_k(τ_ρ)||_F ≈ 2×10^{{-9}}\n")
 f.write("This is below the numerical floor (~1e-10), so the off-line zeros do not\n")
 f.write("perturb the matrix in a detectable way at this basis size.\n\n")
 f.write("MISSING LEMMA: To detect GRH violation via the Weil matrix, the test function\n")
 f.write("basis must contain functions g_j whose Fourier transforms are CONCENTRATED near\n")
 f.write("the heights γ of the off-line zeros (γ ~ 114-5000 here). Such localized bases\n")
 f.write("are computable but require high-precision integration to avoid numerical\n")
 f.write("instability (we attempted with off-line-centered bumps but obtained numerical\n")
 f.write("artifacts producing apparent negative eigenvalues for ALL L-functions, including\n")
 f.write("GRH-satisfying ones).\n\n")
 f.write("L(Δ,s) and ζ_δ: zero data NOT available in workspace.\n")

print("Saved.")
print(open('weil_positivity_results.txt').read())


Saved.
Weil Positivity Test Results

Test functions: Paley-Wiener basis with compactly supported Fourier transforms
 ĝ_j(u) = (1-|u|/X) · cos(jπu/X) on [-X,X], j=0..9, X=4

Method: Computed Weil matrix M_jk = Arch_jk + Pole_jk - Prime_jk
 using arithmetic side of explicit formula (no zero list needed for ζ, L(χ_5), L_DH
 matrix construction itself).

L-functions tested:
 ζ: conductor 1, gamma factor Γ(s/2), pole at s=1 (Λ_ζ on prime powers)
 L(χ_5): conductor 5, gamma factor Γ(s/2) (real quadratic char, EVEN)
 L_DH: conductor 5, gamma factor Γ((s+1)/2) (no Euler product, 110 off-line zeros)

Minimum eigenvalues:
 λ_min(M_ζ) = -6.509e-10
 λ_min(M_χ_5) = -7.985e-10
 λ_min(M_DH) = -1.723e-10

CONCLUSION: All three matrices have minimum eigenvalue ≈ 0 within numerical
precision (~1e-10 from quadrature error). NO function shows a negative λ_min
at this basis size and choice. The hypothesis (M_DH should have negative λ_min)
is NOT reproduced.

Off-line contribution to M_DH (from 110 off-line